<a href="https://colab.research.google.com/github/mtalafha90/CNN_CPC/blob/codex/b48-global-conditioned/notebook/b48_global_conditioned_sparse_mil_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Standalone B48-shaped global-query-conditioned sparse-MIL knee MRI sandbox

This is a separate, self-contained Google Colab notebook for the same small
MRI subset already stored on Google Drive. It preserves the earlier notebook's
bounded DICOM pipeline and native-aspect 448×448 geometry, then replaces the
old global-plus-local model with a matched B48-shaped comparison:

```text
same encoder and same train/validation split
      ├── static_prior_control
      │     pathology query before study-series cross-attention
      └── post_cross_attention_candidate
            pathology query after study-series cross-attention

both → detached 96-dimensional query/token cosine compatibility
     → target-specific 6×6 top-k local evidence
     → zero-start residual fusion with global logits
```

The notebook trains both arms from byte-identical initialization, uses the same
fixed split and data order for each arm, and saves their paired subset result.
It starts with a no-update preflight that checks the zero-start gates,
detachment boundary, and memory footprint before either optimizer takes a step.

## What this notebook does—and does not—mean

This notebook reflects the **B48 mechanism**: a pathology-specific global
query softly re-ranks local spatial tokens through

\[
e_t(x_i) + \tanh(a_t)\,\cos\!\left(W_q\operatorname{LN}(q_t),
W_k\operatorname{LN}(x_i)\right).
\]

`a_t` starts at zero for every target. Therefore each arm begins as ordinary
sparse MIL; the global-context term can influence token ranking only after the
gate learns to open. The global query is detached before it reaches the local
head, so the local auxiliary loss cannot train the global query branch through
that route.

This is a **compact subset sandbox**, not the official B48 result or a
replacement for its protocol. It creates fresh compact weights from the subset
CSV labels, has no Phase-9/B34 checkpoint, no report-only fill artifact, no
official-gold exclusion audit, and no frozen scanner-domain split. Do not use
its hold-out AUC to claim a B48 endpoint, choose a scientific winner, or change
the already prepared full B48 design. Its purpose is to verify the data path,
resource behavior, paired-arm logic, and global-to-spatial representation on
the same Drive subset before the actual B48 protocol is run.

## 1. Google Drive layout

Keep the same two supplied archives at the top level of Google Drive. This new
notebook mounts Drive, copies them to its own local Colab folder, safely unzips
them, and writes only to a new B48-specific output folder. It never edits the
earlier notebook or its `knee_mri_subset_outputs/` results.

```text
MyDrive/
├── colab_subset.zip                         # same labelled training subset
├── test.zip                                 # same unlabelled test subset
├── knee_mri_subset_outputs/                 # earlier notebook; untouched
└── knee_mri_b48_subset_outputs/             # created only by this notebook
```

Expected extracted training layout: `train.csv`, `train_series.csv`, and either
`train_series/` or `train_images/`. Expected extracted test layout: `test.csv`,
`test_series.csv`, and either `test_series/` or `test_images/`.

The train table must contain `StudyInstanceUID` plus the 12 binary target
columns. A target cell can be `0`, `1`, or blank; blank cells are excluded from
the masked weighted BCE loss. The test subset is used only after both arms
finish to produce one prediction CSV per arm.

## 2. Install the DICOM reader

In [ ]:
# Import Python's package installer helper.
import sys
# Import the process runner used to install the missing package.
import subprocess

# Install pydicom because Colab does not guarantee that it is preinstalled.
subprocess.run(
    # Use the current notebook Python interpreter for a compatible installation.
    [sys.executable, "-m", "pip", "install", "-q", "pydicom>=2.4"],
    # Stop this cell immediately if installation fails.
    check=True,
)

## 3. Imports, labels, and reproducibility

In [ ]:
# Enable modern type annotations in every definition in this cell.
from __future__ import annotations

# Import dataclass helpers for clear configuration and experiment containers.
from dataclasses import asdict, dataclass, field
# Import a no-op context manager for CPU execution.
from contextlib import nullcontext
# Import Path for safe cross-platform file paths.
from pathlib import Path
# Import type names used in function annotations.
from typing import Iterable
# Import garbage collection for releasing large CPU tensors between epochs.
import gc
# Import JSON for readable configuration and history files.
import json
# Import math for sine/cosine position features and log-mean-exp pooling.
import math
# Import random for reproducible Python-level sampling.
import random
# Import Linux resource statistics so Colab preflight reports host-RAM pressure too.
import resource
# Import shutil for copying the two Drive archives to Colab's local SSD.
import shutil
# Import time for per-epoch timing.
import time
# Import zipfile for safe archive inspection and extraction.
import zipfile

# Import matplotlib for the loss curve and case-review figures.
import matplotlib.pyplot as plt
# Import NumPy for numerical arrays and deterministic splitting.
import numpy as np
# Import pandas for CSV tables and summary tables.
import pandas as pd
# Import PyTorch's main namespace.
import torch
# Import neural-network layers and functional operations.
import torch.nn.functional as F
from torch import nn
# Import the dataset and loader interfaces.
from torch.utils.data import DataLoader, Dataset
# Import activation checkpointing to trade extra compute for substantially lower GPU memory.
from torch.utils.checkpoint import checkpoint
# Import display so summary tables render in Colab.
from IPython.display import display

# List target columns exactly as they appear in train.csv.
TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]
# Store the target count once so model classes do not repeat a magic number.
N_TARGETS = len(TARGETS)
# Convert normalized plane names into small categorical identifiers.
PLANE_TO_ID = {"Sagittal": 1, "Coronal": 2, "Axial": 3}


def set_seed(seed: int = 2026) -> None:
    """Make splitting and new-model initialization reproducible."""
    # Seed Python's random-number generator.
    random.seed(seed)
    # Seed NumPy's random-number generator.
    np.random.seed(seed)
    # Seed CPU PyTorch operations.
    torch.manual_seed(seed)
    # Seed every visible CUDA device when a GPU is present.
    torch.cuda.manual_seed_all(seed)
    # Prefer repeatability over cuDNN auto-tuned speed.
    torch.backends.cudnn.benchmark = False
    # Request deterministic cuDNN kernels when available.
    torch.backends.cudnn.deterministic = True


# Set the notebook-wide seed before creating any model or split.
set_seed()
# Choose the GPU if Colab provides one; otherwise keep the notebook functional on CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Print the selected device for a quick environment check.
print("device:", DEVICE)
# Print the concrete GPU name when CUDA is available.
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
@dataclass(frozen=True)
class B48SubsetReference:
    """Record the immutable scope boundary for this separate Colab sandbox."""

    experiment: str = "B48-shaped matched global-conditioned sparse-MIL subset sandbox"
    model_status: str = "fresh compact subset weights; no Phase-9/B34 checkpoint"
    arms: tuple[str, str] = (
        "static_prior_control",
        "post_cross_attention_candidate",
    )
    context_dim: int = 96
    context_metric: str = "cosine_low_rank_query_token_compatibility"
    query_gradient: str = "detached_before_local_head"
    official_b48_protocol: str = (
        "not run here: report-only weak labels, gold exclusion, and scanner-domain split are absent"
    )
    interpretation: str = "mechanism and resource check only; not an official B48 gate"


B48_SUBSET_REFERENCE = B48SubsetReference()


def display_b48_subset_reference() -> pd.DataFrame:
    """Display the subset-sandbox boundary before data and model construction."""
    table = pd.DataFrame([asdict(B48_SUBSET_REFERENCE)])
    display(table)
    return table


B48_SUBSET_REFERENCE_TABLE = display_b48_subset_reference()

## 4. Mount Drive, copy both archives locally, and define the run configuration

In [ ]:
def mount_drive(mount_point: str = "/content/drive") -> Path:
    """Mount Google Drive and return the mounted root directory."""
    # Import Colab's Drive helper only inside this Colab-specific function.
    from google.colab import drive
    # Ask Colab to mount the authenticated user's Drive at the requested location.
    drive.mount(mount_point)
    # Return a Path object so later cells use safe path joins.
    return Path(mount_point)


@dataclass(frozen=True)
class ArchivePaths:
    """Locations of the two Drive archives and their fast local extraction folder."""

    # Hold the archive containing train.csv, train_series.csv, and training DICOM files.
    training_archive: Path
    # Hold the archive containing test.csv, test_series.csv, and test DICOM files.
    test_archive: Path
    # Hold the Colab-local folder used while reading large DICOM files.
    local_root: Path


def safe_extract_zip(archive: Path, destination: Path) -> None:
    """Extract one ZIP archive while refusing paths that escape the destination folder."""
    # Create the local destination if this is the first notebook run.
    destination.mkdir(parents=True, exist_ok=True)
    # Resolve the destination once for reliable ZIP member safety checks.
    resolved_destination = destination.resolve()
    # Open the archive for a read-only member inspection and extraction.
    with zipfile.ZipFile(archive) as zip_file:
        # Check every archived filename before writing any file.
        for member in zip_file.infolist():
            # Resolve the output path implied by this archive member.
            target = (destination / member.filename).resolve()
            # Reject an archive member that would write outside the intended folder.
            if target != resolved_destination and resolved_destination not in target.parents:
                raise RuntimeError(f"Unsafe ZIP path in {archive.name}: {member.filename}")
        # Extract all verified files into Colab's local SSD-backed storage.
        zip_file.extractall(destination)


def copy_and_extract_archives(archives: ArchivePaths) -> Path:
    """Copy Drive ZIP files to local storage, then unpack both before any DICOM reading."""
    # Create the local root without deleting an earlier extraction.
    archives.local_root.mkdir(parents=True, exist_ok=True)
    # Process training and test archives in a fixed, readable order.
    for source_archive in (archives.training_archive, archives.test_archive):
        # Stop with the exact missing Drive filename if an archive is not present.
        if not source_archive.is_file():
            raise FileNotFoundError(f"Missing Drive archive: {source_archive}")
        # Put a local copy beside the extracted data using the original filename.
        local_archive = archives.local_root / source_archive.name
        # Copy once from Drive so later DICOM reads avoid the slower mounted filesystem.
        shutil.copy2(source_archive, local_archive)
        # Extract the locally copied archive into the same local root.
        safe_extract_zip(local_archive, archives.local_root)
        # Report the completed copy/extract step.
        print(f"Ready: {source_archive.name} -> {archives.local_root}")
    # Return the local root used by the next path-discovery functions.
    return archives.local_root


def find_extracted_root(local_root: Path, table_name: str, series_table_name: str) -> Path:
    """Find exactly one extracted data root containing both required CSV tables."""
    # Find every parent folder that contains the requested study-level table.
    candidates = [
        path.parent
        for path in local_root.rglob(table_name)
        if (path.parent / series_table_name).is_file()
    ]
    # Stop when archives did not extract the expected pair of tables.
    if not candidates:
        raise FileNotFoundError(
            f"Could not find {table_name} and {series_table_name} below {local_root}"
        )
    # Stop rather than guessing if archives contain more than one matching dataset folder.
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one folder with {table_name} and {series_table_name}; found {candidates}"
        )
    # Return the unambiguous extracted dataset folder.
    return candidates[0]


@dataclass(frozen=True)
class DrivePaths:
    """Locations for local training data and persistent Drive outputs."""

    # Hold the local folder containing train.csv, train_series.csv, and training DICOM files.
    data_root: Path
    # Hold the Drive folder where this notebook writes persistent new results.
    output_root: Path

    @property
    def train_csv(self) -> Path:
        """Return the path of the study-level CSV file."""
        # Join the dataset root and the fixed study-table filename.
        return self.data_root / "train.csv"

    @property
    def series_csv(self) -> Path:
        """Return the path of the series-level CSV file."""
        # Join the dataset root and the fixed series-table filename.
        return self.data_root / "train_series.csv"


def make_paths(dataset_root: str | Path, output_root: str | Path | None = None) -> DrivePaths:
    """Build training-data and output paths from extracted local data and an optional Drive output folder."""
    # Convert either a string or Path into a Path object.
    root = Path(dataset_root)
    # Use a local outputs folder only when no persistent Drive output folder was supplied.
    results = root / "outputs" if output_root is None else Path(output_root)
    # Return the complete training input and output path bundle.
    return DrivePaths(data_root=root, output_root=results)


@dataclass(frozen=True)
class TestPaths:
    """Locations for the separately extracted test subset."""

    # Hold the local folder containing test.csv, test_series.csv, and test DICOM files.
    data_root: Path

    @property
    def test_csv(self) -> Path:
        """Return the test study table path."""
        # Join the test root and fixed test-table filename.
        return self.data_root / "test.csv"

    @property
    def series_csv(self) -> Path:
        """Return the test series metadata table path."""
        # Join the test root and fixed test-series table filename.
        return self.data_root / "test_series.csv"


def make_test_paths(dataset_root: str | Path) -> TestPaths:
    """Build test-subset paths from the extracted local folder."""
    # Convert either a string or Path into a Path object.
    root = Path(dataset_root)
    # Return the complete test input path bundle.
    return TestPaths(data_root=root)


@dataclass(frozen=True)
class RunConfig:
    """All user-adjustable choices for one standalone training run."""

    # Keep the high-resolution in-plane representation.
    image_size: int = 448
    # Use 32 deterministic slice centers per MRI series.
    slices_per_series: int = 32
    # Use immediate neighbors as the other two 2.5D channels.
    triplet_gap: int = 1
    # Retain the central 90 percent of each native-resolution image.
    crop_fraction: float = 0.90
    # Preserve each cropped matrix's aspect ratio while fitting it into the square canvas.
    resize_policy: str = "aspect_preserving_pad"
    # Use normalized black pixels for the symmetric margins added after resize-to-fit.
    pad_value: float = 0.0
    # Pool local encoder features into a 6 by 6 evidence grid.
    grid_size: int = 6
    # Retain the top eight local evidence tokens for every target.
    top_k: int = 8
    # Use a compact 128-dimensional model for practical Colab memory use.
    feature_dim: int = 128
    # Keep B48's fixed low-rank query/token compatibility dimension.
    context_dim: int = 96
    # Use one compact global series-memory transformer layer in this subset sandbox.
    global_memory_layers: int = 1
    # Split the 128-dimensional global representation into four attention heads.
    global_attention_heads: int = 4
    # Apply modest, paired dropout inside the compact global-memory branch.
    global_dropout: float = 0.10
    # Encode one 448-pixel triplet at a time to minimize the peak activation footprint.
    encoder_chunk_size: int = 1
    # Recompute encoder activations during backward instead of retaining every triplet's activations.
    gradient_checkpointing: bool = True
    # Keep four series by default; set a larger value only after that exact setting passes preflight.
    max_series_per_study: int = 4
    # Keep one study per batch to avoid padding and duplicate CPU allocations.
    batch_size: int = 1
    # Decode in the main process so worker processes cannot multiply host RAM use.
    num_workers: int = 0
    # Cap the deterministic DICOM-pixel sample used for memory-bounded percentile normalization.
    percentile_sample_cap: int = 262_144
    # Reserve twenty percent of usable labelled studies for validation.
    validation_fraction: float = 0.20
    # Run a fixed two-epoch matched subset duration; this is not the full B48 protocol.
    epochs: int = 2
    # Set the AdamW learning rate.
    learning_rate: float = 1e-4
    # Set the AdamW weight decay.
    weight_decay: float = 1e-4
    # Give the sparse local classifier a direct auxiliary loss.
    local_loss_weight: float = 1.0
    # Limit one optimizer update's gradient norm.
    grad_clip_norm: float = 1.0
    # Save the split and initialization randomness with this seed.
    seed: int = 2026
    # Raise on a bad DICOM when true; otherwise skip just that unreadable series.
    strict_dicom: bool = False


# Mount the user's Google Drive once for this notebook session.
DRIVE_ROOT = mount_drive()
# Declare the two archive names supplied by the user and the fast local extraction folder.
ARCHIVES = ArchivePaths(
    training_archive=DRIVE_ROOT / "MyDrive" / "colab_subset.zip",
    test_archive=DRIVE_ROOT / "MyDrive" / "test.zip",
    local_root=Path("/content/knee_mri_b48_subset"),
)
# Copy and unpack both archives before any training or test DICOM read.
LOCAL_ROOT = copy_and_extract_archives(ARCHIVES)
# Find the extracted training root without assuming how the ZIP file nests its top folder.
TRAINING_ROOT = find_extracted_root(LOCAL_ROOT, "train.csv", "train_series.csv")
# Find the extracted test root without assuming how the ZIP file nests its top folder.
TEST_ROOT = find_extracted_root(LOCAL_ROOT, "test.csv", "test_series.csv")
# Read high-volume training images from local storage and save results persistently to Drive.
PATHS = make_paths(TRAINING_ROOT, DRIVE_ROOT / "MyDrive" / "knee_mri_b48_subset_outputs")
# Read high-volume test images from local storage.
TEST_PATHS = make_test_paths(TEST_ROOT)
# Create the default conservative training configuration.
CONFIG = RunConfig()
# Display the resolved training paths for confirmation.
print(PATHS)
# Display the resolved test paths for confirmation.
print(TEST_PATHS)
# Display all active model and memory settings for confirmation.
print(CONFIG)

## 5. Validate the tables and create the MRI series index

In [ ]:
# Define accepted text values that mean true in the metadata table.
TRUE_TOKENS = {"true", "t", "yes", "y", "1", "1.0"}
# Define accepted text values that mean false in the metadata table.
FALSE_TOKENS = {"false", "f", "no", "n", "0", "0.0"}


def parse_bool(value: object) -> int:
    """Convert a metadata flag to 0 unknown, 1 false, or 2 true."""
    # Preserve missing metadata as the unknown code.
    if pd.isna(value):
        return 0
    # Convert Python and NumPy booleans directly.
    if isinstance(value, (bool, np.bool_)):
        return 2 if bool(value) else 1
    # Convert numeric nonzero values into true and zero values into false.
    if isinstance(value, (int, float, np.integer, np.floating)):
        return 2 if float(value) != 0 else 1
    # Normalize text before checking the accepted tokens.
    text = str(value).strip().lower()
    # Map accepted true tokens to code two.
    if text in TRUE_TOKENS:
        return 2
    # Map accepted false tokens to code one.
    if text in FALSE_TOKENS:
        return 1
    # Treat all other values as unknown.
    return 0


def normalise_plane(value: object) -> str:
    """Map common plane spelling variants to a standard anatomical-plane name."""
    # Define the accepted spelling variants.
    mapping = {
        "sagittal": "Sagittal", "sag": "Sagittal", "sagital": "Sagittal",
        "coronal": "Coronal", "cor": "Coronal",
        "axial": "Axial", "ax": "Axial", "transverse": "Axial",
    }
    # Return an empty string for a plane that cannot be recognized safely.
    return mapping.get(str(value).strip().lower(), "")


def validate_dataset(paths: DrivePaths) -> dict:
    """Validate the local training CSV schema and DICOM layout before using the GPU."""
    # List the two CSV files required by the standalone workflow.
    required_files = (paths.train_csv, paths.series_csv)
    # Collect every missing CSV filename so one error explains the whole problem.
    missing_files = [str(path) for path in required_files if not path.is_file()]
    # Stop before loading data if a required file is absent.
    if missing_files:
        raise FileNotFoundError("Missing required file(s):\n" + "\n".join(missing_files))
    # Load the study-level table.
    train = pd.read_csv(paths.train_csv)
    # Load the series-level table.
    series = pd.read_csv(paths.series_csv)
    # Require a study UID and every classification target in the study table.
    train_required = {"StudyInstanceUID", *TARGETS}
    # Require series identifiers and routing metadata in the series table.
    series_required = {
        "StudyInstanceUID", "SeriesInstanceUID", "Fluid_Sensitive",
        "Fat_Suppression", "Anatomical_Plane",
    }
    # Find missing study-table columns.
    missing_train = sorted(train_required.difference(train.columns))
    # Find missing series-table columns.
    missing_series = sorted(series_required.difference(series.columns))
    # Explain an incomplete study table explicitly.
    if missing_train:
        raise ValueError(f"train.csv missing columns: {missing_train}")
    # Explain an incomplete series table explicitly.
    if missing_series:
        raise ValueError(f"train_series.csv missing columns: {missing_series}")
    # Copy tables before normalizing their identifiers.
    train = train.copy()
    # Convert study IDs into strings so numeric-looking UIDs do not lose leading digits.
    train["StudyInstanceUID"] = train["StudyInstanceUID"].astype(str)
    # Convert series study IDs into strings for the same reason.
    series["StudyInstanceUID"] = series["StudyInstanceUID"].astype(str)
    # Convert series IDs into strings for safe directory lookup.
    series["SeriesInstanceUID"] = series["SeriesInstanceUID"].astype(str)
    # Reject duplicate studies because each study needs exactly one target row.
    if train["StudyInstanceUID"].duplicated().any():
        raise ValueError("train.csv contains duplicate StudyInstanceUID values")
    # Reject duplicate series entries because one directory should map to one metadata row.
    if series[["StudyInstanceUID", "SeriesInstanceUID"]].duplicated().any().any():
        raise ValueError("train_series.csv contains duplicate study/series rows")
    # Convert every target to numeric while preserving blank cells as NaN.
    labels = train[TARGETS].apply(pd.to_numeric, errors="coerce")
    # Record which target cells have a usable CSV label.
    known = labels.notna()
    # Identify labels that are neither zero nor one.
    invalid = known & ~labels.isin([0.0, 1.0])
    # Stop on invalid label values so the loss never silently interprets a bad code.
    if invalid.any().any():
        bad = invalid.sum()[invalid.sum() > 0].to_dict()
        raise ValueError(f"Target values must be 0, 1, or blank; invalid counts: {bad}")
    # Stop if the selected subset contains no supervised cells at all.
    if not known.any().any():
        raise ValueError("The subset has no known target labels in train.csv")
    # Normalize plane text before reporting how many MRI series can be used.
    plane = series["Anatomical_Plane"].map(normalise_plane)
    # Mark rows that have one of the three recognized anatomical planes.
    eligible = plane.isin(PLANE_TO_ID)
    # Accept either original-style DICOM root name.
    roots = [paths.data_root / "train_series", paths.data_root / "train_images"]
    # Stop with a clear path error if neither DICOM root exists.
    if not any(root.is_dir() for root in roots):
        raise FileNotFoundError(
            "Expected train_series/ or train_images/ under " f"{paths.data_root}"
        )
    # Build a concise data audit that is useful to save or screenshot.
    result = {
        "studies": int(len(train)),
        "series_rows": int(len(series)),
        "recognized_plane_series": int(eligible.sum()),
        "studies_with_any_label": int(known.any(axis=1).sum()),
        "known_label_cells": int(known.to_numpy().sum()),
        "data_root": str(paths.data_root),
    }
    # Print the audit in a readable JSON form.
    print(json.dumps(result, indent=2))
    # Return the audit for optional downstream use.
    return result


def validate_test_dataset(paths: TestPaths) -> dict:
    """Validate the extracted test CSV schema and DICOM layout before inference."""
    # List the two test CSV files required for prediction.
    required_files = (paths.test_csv, paths.series_csv)
    # Collect all missing test files for one actionable error message.
    missing_files = [str(path) for path in required_files if not path.is_file()]
    # Stop before model inference if the archive omitted a required file.
    if missing_files:
        raise FileNotFoundError("Missing required test file(s):\n" + "\n".join(missing_files))
    # Load the test study table.
    test = pd.read_csv(paths.test_csv)
    # Load the test series metadata table.
    series = pd.read_csv(paths.series_csv)
    # Require a unique study identifier in the test study table.
    test_required = {"StudyInstanceUID"}
    # Require the same MRI-routing metadata used by the training data.
    series_required = {
        "StudyInstanceUID", "SeriesInstanceUID", "Fluid_Sensitive",
        "Fat_Suppression", "Anatomical_Plane",
    }
    # Find missing test-study columns.
    missing_test = sorted(test_required.difference(test.columns))
    # Find missing test-series columns.
    missing_series = sorted(series_required.difference(series.columns))
    # Explain a malformed test study table.
    if missing_test:
        raise ValueError(f"test.csv missing columns: {missing_test}")
    # Explain a malformed test series table.
    if missing_series:
        raise ValueError(f"test_series.csv missing columns: {missing_series}")
    # Normalize identifiers before checking their uniqueness.
    test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
    # Normalize test-series study identifiers.
    series["StudyInstanceUID"] = series["StudyInstanceUID"].astype(str)
    # Normalize test-series identifiers.
    series["SeriesInstanceUID"] = series["SeriesInstanceUID"].astype(str)
    # Reject duplicate test studies because predictions need one row per study.
    if test["StudyInstanceUID"].duplicated().any():
        raise ValueError("test.csv contains duplicate StudyInstanceUID values")
    # Reject duplicate test series metadata rows.
    if series[["StudyInstanceUID", "SeriesInstanceUID"]].duplicated().any().any():
        raise ValueError("test_series.csv contains duplicate study/series rows")
    # Find recognized anatomical plane metadata rows.
    eligible = series["Anatomical_Plane"].map(normalise_plane).isin(PLANE_TO_ID)
    # Accept either original-style test DICOM root name.
    roots = [paths.data_root / "test_series", paths.data_root / "test_images"]
    # Stop before inference if no test DICOM hierarchy was extracted.
    if not any(root.is_dir() for root in roots):
        raise FileNotFoundError(
            "Expected test_series/ or test_images/ under " f"{paths.data_root}"
        )
    # Build a concise test-subset audit.
    result = {
        "test_studies": int(len(test)),
        "test_series_rows": int(len(series)),
        "test_recognized_plane_series": int(eligible.sum()),
        "test_data_root": str(paths.data_root),
    }
    # Print the test audit in readable JSON.
    print(json.dumps(result, indent=2))
    # Return the test audit for optional later use.
    return result


def build_series_records(series: pd.DataFrame, config: RunConfig) -> dict[str, list[dict]]:
    """Create an ordered list of usable MRI series for each study."""
    # Work on a copy so callers keep their original table unchanged.
    work = series.copy()
    # Normalize study UIDs for dictionary keys.
    work["StudyInstanceUID"] = work["StudyInstanceUID"].astype(str)
    # Normalize series UIDs for directory lookup.
    work["SeriesInstanceUID"] = work["SeriesInstanceUID"].astype(str)
    # Normalize anatomical-plane text.
    work["plane"] = work["Anatomical_Plane"].map(normalise_plane)
    # Map each recognized plane to its categorical code.
    work["plane_id"] = work["plane"].map(PLANE_TO_ID).fillna(0).astype(int)
    # Map fluid sensitivity to its categorical code.
    work["fluid_id"] = work["Fluid_Sensitive"].map(parse_bool).astype(int)
    # Map fat suppression to its categorical code.
    work["fat_id"] = work["Fat_Suppression"].map(parse_bool).astype(int)
    # Discard unrecognized planes because their spatial orientation is unknown.
    work = work.loc[work["plane_id"] > 0].copy()
    # Prepare the final study-to-series mapping.
    result: dict[str, list[dict]] = {}
    # Build one deterministic record list per study.
    for uid, part in work.groupby("StudyInstanceUID", sort=False):
        # Convert selected metadata columns into plain dictionaries.
        rows = [
            {
                "series_uid": str(row.SeriesInstanceUID),
                "plane": str(row.plane),
                "plane_id": int(row.plane_id),
                "fluid_id": int(row.fluid_id),
                "fat_id": int(row.fat_id),
            }
            for row in part.itertuples(index=False)
        ]
        # Keep ordering reproducible and avoid an arbitrary filesystem order.
        rows.sort(
            key=lambda row: (
                row["plane_id"], row["fluid_id"], row["fat_id"], row["series_uid"]
            )
        )
        # Limit the number of series only when the configured limit is positive.
        if config.max_series_per_study > 0:
            rows = rows[: config.max_series_per_study]
        # Store only studies that retain at least one usable series.
        if rows:
            result[str(uid)] = rows
    # Return the mapping consumed by the dataset class.
    return result


# Run the table and directory audit immediately after setting the paths.
DATASET_SUMMARY = validate_dataset(PATHS)
# Run the test-table and directory audit immediately after setting the test paths.
TEST_DATASET_SUMMARY = validate_test_dataset(TEST_PATHS)

## 6. DICOM decoding and 448×448 2.5D preparation

In [ ]:
# Accept normal DICOM suffixes and files without a suffix.
DICOM_SUFFIXES = {"", ".dcm", ".dicom", ".ima"}


def find_series_dir(data_root: Path, split: str, study_uid: str, series_uid: str) -> Path | None:
    """Locate one train or test DICOM series in either accepted directory hierarchy."""
    # Reject an unexpected split name before constructing a filesystem path.
    if split not in {"train", "test"}:
        raise ValueError(f"split must be 'train' or 'test', got {split!r}")
    # Check both accepted DICOM root names in a fixed order for this split.
    for root_name in (f"{split}_series", f"{split}_images"):
        # Build the expected study/series directory path.
        candidate = data_root / root_name / str(study_uid) / str(series_uid)
        # Return immediately when the expected directory exists.
        if candidate.is_dir():
            return candidate
    # Return None when no expected directory exists.
    return None


def dicom_sort_key(dataset) -> float:
    """Use patient-space slice position when available, else use instance number."""
    # Try the geometric DICOM ordering first.
    try:
        # Read the DICOM image position vector.
        position = np.asarray(dataset.ImagePositionPatient, dtype=float)
        # Read the DICOM row and column direction vectors.
        orientation = np.asarray(dataset.ImageOrientationPatient, dtype=float)
        # Project position onto the slice normal to get physical ordering.
        return float(np.dot(position, np.cross(orientation[:3], orientation[3:])))
    # Fall back safely when geometric fields are missing or malformed.
    except Exception:
        # Use InstanceNumber as the fallback ordering key.
        return float(getattr(dataset, "InstanceNumber", 0))


def center_pad_to_shape(image: np.ndarray, target_shape: tuple[int, int]) -> np.ndarray:
    """Centre-pad one unexpected mixed-matrix frame without discarding native pixels."""
    # Reject a target smaller than the source because this safety fallback never crops images.
    if image.shape[0] > target_shape[0] or image.shape[1] > target_shape[1]:
        raise ValueError(f"Cannot pad {image.shape} into smaller target {target_shape}")
    # Allocate NaN margins so later normalization can replace only synthetic padding.
    output = np.full(target_shape, np.nan, dtype=np.float32)
    # Keep every original source row.
    rows = image.shape[0]
    # Keep every original source column.
    cols = image.shape[1]
    # Centre the source rows inside the larger common matrix.
    dst_r = (target_shape[0] - rows) // 2
    # Centre the source columns inside the larger common matrix.
    dst_c = (target_shape[1] - cols) // 2
    # Copy all native pixels with no crop and no interpolation.
    output[dst_r : dst_r + rows, dst_c : dst_c + cols] = image
    # Return the padded single frame.
    return output


@dataclass(frozen=True)
class DicomFrameReference:
    """Point to one frame without retaining its DICOM pixel matrix in RAM."""

    # Store the DICOM file that contains this frame.
    path: Path
    # Store the zero-based frame index inside a multi-frame file.
    frame_index: int
    # Store the deterministic physical ordering key.
    sort_key: float
    # Store the header-declared native matrix shape.
    shape: tuple[int, int]


def list_dicom_frame_references(series_dir: Path) -> list[DicomFrameReference]:
    """Read DICOM headers only and return ordered frame references with bounded RAM use."""
    # Import pydicom inside the function so the notebook imports cleanly before installation.
    import pydicom
    # Collect eligible DICOM files in deterministic filename order.
    files = sorted(
        path for path in series_dir.iterdir()
        if path.is_file() and path.suffix.lower() in DICOM_SUFFIXES
    )
    # Prepare a compact list that holds metadata but no image arrays.
    references: list[DicomFrameReference] = []
    # Count header failures for an informative error message.
    failures = 0
    # Read each header without loading its potentially large PixelData element.
    for path in files:
        # Handle a malformed header independently so another valid slice can still be used.
        try:
            # Read only metadata; stop before the image payload to keep host RAM bounded.
            dataset = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
            # Read the geometry or instance-number sorting key.
            key = dicom_sort_key(dataset)
            # Read the header matrix dimensions required for the mixed-matrix safety fallback.
            shape = (int(dataset.Rows), int(dataset.Columns))
            # Read the number of frames while treating an ordinary DICOM as one frame.
            count = int(getattr(dataset, "NumberOfFrames", 1))
            # Reject invalid frame counts before constructing references.
            if count < 1:
                raise RuntimeError(f"Invalid NumberOfFrames={count}")
            # Add one lightweight reference for every frame in this DICOM file.
            for frame_index in range(count):
                # Offset equal-position multi-frame images into a deterministic within-file order.
                references.append(
                    DicomFrameReference(path, frame_index, key + frame_index * 1e-4, shape)
                )
        # Count the bad header and continue scanning the remaining files.
        except Exception:
            failures += 1
    # Stop if no candidate frame had a readable header.
    if not references:
        raise RuntimeError(
            f"No readable DICOM headers in {series_dir} "
            f"({len(files)} files, {failures} header failures)"
        )
    # Sort references along the physical MRI acquisition direction.
    references.sort(key=lambda reference: reference.sort_key)
    # Return only compact metadata, never an in-memory full MRI volume.
    return references


def decode_dicom_frame(reference: DicomFrameReference) -> np.ndarray:
    """Decode exactly one referenced DICOM frame as a native float32 image."""
    # Import pydicom locally so this helper is self-contained in Colab.
    import pydicom
    # Read the one DICOM file that contains the requested frame.
    dataset = pydicom.dcmread(str(reference.path), force=True)
    # Decode its pixel payload; a multi-frame file is still decoded only when encountered.
    decoded = np.asarray(dataset.pixel_array)
    # Select a normal single-frame image when the file contains one two-dimensional matrix.
    if decoded.ndim == 2:
        # Guard against a malformed reference that asks for a non-existent second frame.
        if reference.frame_index != 0:
            raise RuntimeError(f"Single-frame DICOM requested frame {reference.frame_index}")
        # Keep the two-dimensional decoded matrix.
        pixels = decoded
    # Select the requested frame from a three-dimensional multi-frame DICOM.
    elif decoded.ndim == 3:
        # Guard against a DICOM whose header and decoded frame count disagree.
        if reference.frame_index >= decoded.shape[0]:
            raise RuntimeError(f"Multi-frame DICOM has only {decoded.shape[0]} frames")
        # Keep exactly the requested two-dimensional frame.
        pixels = decoded[reference.frame_index]
    # Refuse unsupported DICOM pixel dimensionality explicitly.
    else:
        raise RuntimeError(f"Unsupported decoded DICOM shape: {decoded.shape}")
    # Convert the retained native frame to float32 before intensity rescaling.
    pixels = np.asarray(pixels, dtype=np.float32)
    # Apply the DICOM rescale slope when present.
    pixels *= float(getattr(dataset, "RescaleSlope", 1.0))
    # Apply the DICOM rescale intercept when present.
    pixels += float(getattr(dataset, "RescaleIntercept", 0.0))
    # Invert MONOCHROME1 images so bright tissue remains bright.
    if str(getattr(dataset, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
        # Create the conventional bright-tissue orientation without changing the source files.
        pixels = float(np.nanmax(pixels)) - pixels
    # Return a contiguous float32 matrix suitable for NumPy and PyTorch operations.
    return np.ascontiguousarray(pixels, dtype=np.float32)


def deterministic_pixel_sample(image: np.ndarray, count: int) -> np.ndarray:
    """Take a bounded, evenly spaced finite-pixel sample from one native MRI frame."""
    # Flatten only this one frame instead of a whole MRI series.
    flat = np.asarray(image, dtype=np.float32).reshape(-1)
    # Exclude NaNs and infinities before percentile estimation.
    finite = flat[np.isfinite(flat)]
    # Return all values when this frame is already smaller than the requested sample count.
    if finite.size <= count:
        return finite
    # Choose evenly spaced source indices so the sampling is deterministic and reproducible.
    index = np.linspace(0, finite.size - 1, num=count, dtype=np.int64)
    # Return only the bounded representative sample.
    return finite[index]


def streaming_percentile_bounds(
    references: list[DicomFrameReference],
    sample_cap: int,
    retained_indices: set[int],
) -> tuple[float, float, dict[int, np.ndarray]]:
    """Estimate robust series percentiles while retaining only selected native frames."""
    # Reject a nonsensical sample budget before reading any DICOM pixel payload.
    if sample_cap < 1:
        raise ValueError("percentile_sample_cap must be positive")
    # Limit the number of source frames sampled when an unusual series has more frames than the budget.
    sample_count = min(len(references), sample_cap)
    # Choose deterministic frame locations that span the whole acquired series.
    sample_indices = set(
        np.linspace(0, len(references) - 1, num=sample_count, dtype=np.int64).tolist()
    )
    # Allocate the bounded number of pixels contributed by every sampled frame.
    per_frame = max(1, min(4096, sample_cap // max(len(sample_indices), 1)))
    # Keep compact sample fragments rather than a full [frames, height, width] volume.
    samples: list[np.ndarray] = []
    # Keep only the selected frames needed by the later 32 triplets and their neighbors.
    retained_frames: dict[int, np.ndarray] = {}
    # Decode only sampled or later-selected frames one at a time so host RAM stays bounded.
    for index, reference in enumerate(references):
        # Skip frames that neither contribute to normalization nor to a final 2.5D triplet.
        if index not in sample_indices and index not in retained_indices:
            continue
        # Allow an unreadable non-selected frame to be skipped during percentile sampling.
        try:
            # Decode this one native frame.
            frame = decode_dicom_frame(reference)
            # Save its small deterministic intensity sample only when this frame is part of the sample plan.
            if index in sample_indices:
                samples.append(deterministic_pixel_sample(frame, per_frame))
            # Keep the full matrix only when the later 2.5D construction needs it.
            if index in retained_indices:
                retained_frames[index] = frame
            # Release every non-selected image before reading the next DICOM file.
            else:
                del frame
        # Ignore an unreadable frame here; selected frames are checked again before use.
        except Exception:
            continue
    # Stop when no DICOM frame supplied any finite intensity values.
    if not samples:
        raise RuntimeError("No finite DICOM pixels were available for percentile normalization")
    # Join only the bounded representative samples for robust percentile estimation.
    pooled = np.concatenate(samples).astype(np.float32, copy=False)
    # Stop when all decoded images happened to contain only non-finite values.
    if pooled.size == 0:
        raise RuntimeError("DICOM frames contained no finite pixels")
    # Estimate the familiar 1st and 99th percentile bounds without a full-volume allocation.
    low, high = np.percentile(pooled, [1, 99])
    # Ensure a constant-valued series still has a valid nonzero normalization range.
    high = max(float(high), float(low) + 1e-6)
    # Return scalar bounds plus the few native frames that must be reused for triplets.
    return float(low), float(high), retained_frames


def normalize_native_frame(
    frame: np.ndarray,
    low: float,
    high: float,
    target_shape: tuple[int, int],
) -> np.ndarray:
    """Normalize one selected native frame using the series-level streaming bounds."""
    # Pad an unexpected mixed matrix only after percentiles were estimated from real pixels.
    if frame.shape != target_shape:
        frame = center_pad_to_shape(frame, target_shape)
    # Replace synthetic NaNs and unexpected non-finite values with the nearest valid bound.
    normalized = np.nan_to_num(frame, nan=low, posinf=high, neginf=low, copy=True)
    # Move the lower robust intensity bound to zero.
    normalized -= low
    # Scale the robust intensity interval into the zero-to-one range.
    normalized /= max(high - low, 1e-6)
    # Clip remaining outliers into the same normalized support.
    np.clip(normalized, 0.0, 1.0, out=normalized)
    # Return a contiguous float32 native image.
    return np.ascontiguousarray(normalized, dtype=np.float32)


def sample_centers(n_frames: int, n_samples: int, gap: int) -> tuple[np.ndarray, np.ndarray]:
    """Return deterministic slice centres and normalized through-plane positions."""
    # Reject invalid sampling settings before calculating slice indices.
    if n_frames < 1 or n_samples < 1 or gap < 1:
        raise ValueError("frames, samples, and gap must all be positive")
    # Avoid edge centers when the series is long enough for a full 2.5D neighborhood.
    low, high = (gap, n_frames - 1 - gap) if n_frames > 2 * gap else (0, n_frames - 1)
    # Spread the requested centers evenly across the usable MRI range.
    centres = np.round(np.linspace(low, high, n_samples)).astype(np.int64)
    # Normalize the chosen centers into the zero-to-one through-plane coordinate range.
    positions = centres.astype(np.float32) / float(max(n_frames - 1, 1))
    # Return both integer frame indices and continuous positions.
    return centres, positions


def native_center_crop(triplets: np.ndarray, fraction: float) -> np.ndarray:
    """Crop every 2.5D triplet before its one high-resolution resize-to-fit."""
    # Validate the requested retained image fraction.
    if not 0 < fraction <= 1:
        raise ValueError("crop_fraction must be in the interval (0, 1]")
    # Read the native height and width from the final two array dimensions.
    height, width = triplets.shape[-2:]
    # Round the requested native crop height while keeping at least two pixels.
    crop_h = max(2, min(height, int(round(height * fraction))))
    # Round the requested native crop width while keeping at least two pixels.
    crop_w = max(2, min(width, int(round(width * fraction))))
    # Calculate the centered top edge.
    top = (height - crop_h) // 2
    # Calculate the centered left edge.
    left = (width - crop_w) // 2
    # Return the centered native-resolution crop.
    return triplets[..., top : top + crop_h, left : left + crop_w]


def resize_triplets_aspect_preserving_pad(
    triplets: np.ndarray,
    image_size: int,
    pad_value: float,
) -> torch.Tensor:
    """Resize a [triplets, channels, H, W] batch once, then centre-pad it square."""
    # Require the expected 2.5D tensor layout before reading spatial dimensions.
    if triplets.ndim != 4:
        raise ValueError(f"Expected [triplets, channels, height, width], got {triplets.shape}")
    # Require a useful positive target canvas.
    if image_size < 2:
        raise ValueError("image_size must be at least two pixels")
    # Read the retained native crop dimensions.
    height, width = triplets.shape[-2:]
    # Choose one common scale so neither resized side exceeds the square canvas.
    scale = min(image_size / float(height), image_size / float(width))
    # Round the fitted height while keeping it inside the target canvas.
    resized_h = max(1, min(image_size, int(round(height * scale))))
    # Round the fitted width while keeping it inside the target canvas.
    resized_w = max(1, min(image_size, int(round(width * scale))))
    # Convert the contiguous crop into the [batch, channels, H, W] PyTorch layout.
    tensor = torch.from_numpy(np.ascontiguousarray(triplets))
    # Resize every triplet once with antialiased bilinear interpolation.
    resized = F.interpolate(
        tensor,
        size=(resized_h, resized_w),
        mode="bilinear",
        align_corners=False,
        antialias=True,
    )
    # Allocate the fixed square model canvas using normalized black margins.
    output = resized.new_full(
        (resized.shape[0], resized.shape[1], image_size, image_size),
        float(pad_value),
    )
    # Calculate the symmetric vertical margin before copying the resized crop.
    top = (image_size - resized_h) // 2
    # Calculate the symmetric horizontal margin before copying the resized crop.
    left = (image_size - resized_w) // 2
    # Copy the complete resized crop into the centre without a second interpolation.
    output[..., top : top + resized_h, left : left + resized_w] = resized
    # Return the fixed 448-by-448-style tensor with the retained aspect ratio intact.
    return output


def prepare_series_tensor_from_dicom(
    series_dir: Path,
    config: RunConfig,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Stream one series into aspect-preserving [slices, 3, 448, 448] triplets."""
    # Reject an accidental return to direct square stretching before any expensive decoding.
    if config.resize_policy != "aspect_preserving_pad":
        raise ValueError(
            "resize_policy must be 'aspect_preserving_pad' for this native-geometry notebook"
        )
    # List DICOM frame metadata without retaining a full MRI volume in host memory.
    references = list_dicom_frame_references(series_dir)
    # Select deterministic centers and their normalized slice positions from header counts.
    centres, positions = sample_centers(
        len(references), config.slices_per_series, config.triplet_gap
    )
    # Define the previous, central, and next frame offsets for each 2.5D input.
    offsets = np.asarray([-config.triplet_gap, 0, config.triplet_gap], dtype=np.int64)
    # Clip all neighbor indices to valid DICOM frame locations.
    index = np.clip(centres[:, None] + offsets[None, :], 0, len(references) - 1)
    # Record the unique frames that the final 32 triplets will need.
    retained_indices = {int(value) for value in index.reshape(-1)}
    # Estimate series normalization bounds while caching only the selected native frames.
    low, high, cache = streaming_percentile_bounds(
        references,
        config.percentile_sample_cap,
        retained_indices,
    )
    # Build a safety shape only for unexpected mixed-matrix series.
    target_shape = (
        max(reference.shape[0] for reference in references),
        max(reference.shape[1] for reference in references),
    )
    # Allocate only the final fixed-size image stack, not a full native-resolution volume.
    images = torch.empty(
        config.slices_per_series,
        3,
        config.image_size,
        config.image_size,
        dtype=torch.float32,
    )
    # Build one native triplet and one 448-pixel output at a time.
    for output_index, frame_indices in enumerate(index):
        # Prepare the three normalized native frames for this one 2.5D triplet.
        channels: list[np.ndarray] = []
        # Decode or reuse each of the previous, center, and next source frames.
        for frame_index in frame_indices:
            # Convert NumPy's integer type into a normal dictionary key.
            frame_index = int(frame_index)
            # Reuse the bounded cache when this frame was retained during percentile sampling.
            frame = cache.get(frame_index)
            # Decode a selected frame again only when it failed to enter the cache.
            if frame is None:
                frame = decode_dicom_frame(references[frame_index])
            # Normalize this one frame using the same series-level robust intensity bounds.
            channels.append(normalize_native_frame(frame, low, high, target_shape))
        # Stack only the three native channels required for this output triplet.
        triplet = np.stack(channels, axis=0)[None, ...]
        # Crop the triplet in native pixels before its one permitted high-resolution resize.
        cropped = native_center_crop(triplet, config.crop_fraction)
        # Resize-to-fit and center-pad this one triplet into the fixed model canvas.
        resized = resize_triplets_aspect_preserving_pad(
            cropped,
            config.image_size,
            config.pad_value,
        )
        # Copy the completed triplet into its final compact output slot.
        images[output_index].copy_(resized[0])
        # Release the temporary native arrays before the next triplet is constructed.
        del channels, triplet, cropped, resized
    # Drop cached native matrices before the caller starts processing the next MRI series.
    cache.clear()
    # Ask Python to collect temporary DICOM and NumPy objects before returning.
    gc.collect()
    # Return the final 448-pixel image stack and its through-plane coordinates.
    return images, torch.from_numpy(positions)

## 7. Dataset and batch collation classes

In [ ]:
class KneeMRIDataset(Dataset):
    """Decode one train or test study as a variable number of high-resolution MRI series."""

    def __init__(
        self,
        frame: pd.DataFrame,
        series_records: dict[str, list[dict]],
        paths: DrivePaths | TestPaths,
        config: RunConfig,
        split: str,
        include_targets: bool,
    ) -> None:
        # Store file locations for lazy DICOM loading.
        self.paths = paths
        # Store image and error-handling settings.
        self.config = config
        # Store whether DICOM folders and CSV rows belong to the train or test subset.
        self.split = split
        # Store whether this split carries known training labels.
        self.include_targets = include_targets
        # Copy rows so filtering cannot mutate a caller's data frame.
        work = frame.copy()
        # Normalize UID strings to match the series-record dictionary keys.
        work["StudyInstanceUID"] = work["StudyInstanceUID"].astype(str)
        # Keep only studies that have at least one recognized-plane MRI series.
        work = work.loc[work["StudyInstanceUID"].isin(series_records)].reset_index(drop=True)
        # Stop with an actionable error when no usable study remains.
        if work.empty:
            raise ValueError("No studies have a recognized-plane MRI series")
        # Store deterministic study order for DataLoader indexing.
        self.study_uids = work["StudyInstanceUID"].tolist()
        # Store training targets as float32 only for the labelled training split.
        self.targets = (
            work[TARGETS].apply(pd.to_numeric, errors="coerce").to_numpy(np.float32)
            if self.include_targets else None
        )
        # Store the metadata records used to find and describe each MRI series.
        self.series_records = series_records

    def __len__(self) -> int:
        """Return the number of studies in this split."""
        # Let PyTorch know how many valid integer indices exist.
        return len(self.study_uids)

    def _zero_series(self) -> tuple[torch.Tensor, torch.Tensor, float]:
        """Create a shape-compatible placeholder for one unreadable series."""
        # Make a zero image tensor with the same shape as a prepared MRI series.
        images = torch.zeros(
            self.config.slices_per_series,
            3,
            self.config.image_size,
            self.config.image_size,
            dtype=torch.float32,
        )
        # Make zero positions for the unreadable placeholder.
        positions = torch.zeros(self.config.slices_per_series, dtype=torch.float32)
        # Return the placeholder with present=0 so the model masks it out.
        return images, positions, 0.0

    def _load_series(self, study_uid: str, record: dict) -> tuple[torch.Tensor, torch.Tensor, float]:
        """Read and preprocess one MRI series, or return a masked placeholder."""
        # Locate the expected DICOM directory for this study and series.
        series_dir = find_series_dir(
            self.paths.data_root, self.split, study_uid, record["series_uid"]
        )
        # Handle a completely missing series directory.
        if series_dir is None:
            # Raise immediately only when strict DICOM behavior is requested.
            if self.config.strict_dicom:
                raise FileNotFoundError(f"Missing series {study_uid}/{record['series_uid']}")
            # Otherwise let the model ignore a zero placeholder.
            return self._zero_series()
        # Try to decode and prepare the available DICOM directory.
        try:
            # Stream DICOM frames into high-resolution triplets without retaining a full native volume.
            images, positions = prepare_series_tensor_from_dicom(series_dir, self.config)
            # Mark this series readable so the model uses it.
            return images, positions, 1.0
        # Handle a DICOM decode or preprocessing failure.
        except Exception:
            # Raise the original error in strict mode to find the bad input quickly.
            if self.config.strict_dicom:
                raise
            # Otherwise ignore only this series and continue with the study.
            return self._zero_series()

    def __getitem__(self, index: int) -> dict:
        """Load one variable-series study on demand."""
        # Resolve the study identifier from the dataset's deterministic order.
        study_uid = self.study_uids[index]
        # Prepare lists for every real or placeholder MRI series.
        images, positions, present, metadata = [], [], [], []
        # Load every selected series in this study.
        for record in self.series_records[study_uid]:
            # Decode one series and obtain its readable flag.
            image, position, readable = self._load_series(study_uid, record)
            # Append the prepared 2.5D image stack.
            images.append(image)
            # Append matching through-plane positions.
            positions.append(position)
            # Append the model mask flag.
            present.append(readable)
            # Append plane, fluid, and fat-suppression categorical metadata.
            metadata.append([record["plane_id"], record["fluid_id"], record["fat_id"]])
        # Stop if no selected MRI series could be decoded for this study.
        if not any(present):
            raise RuntimeError(f"Study {study_uid} has no readable MRI series")
        # Return tensors with a variable first dimension equal to the series count.
        item = {
            "study_uid": study_uid,
            "volumes": torch.stack(images),
            "slice_position": torch.stack(positions),
            "present": torch.tensor(present, dtype=torch.float32),
            "series_meta": torch.tensor(metadata, dtype=torch.long),
        }
        # Attach labels only when this is the labelled training subset.
        if self.targets is not None:
            item["target"] = torch.from_numpy(self.targets[index])
        # Return the train or test sample dictionary.
        return item


def collate_studies(batch: list[dict]) -> dict:
    """Pad only the variable series dimension when combining studies into a batch."""
    # Reject the impossible empty batch case explicitly.
    if not batch:
        raise ValueError("Cannot collate an empty batch")
    # Use a zero-copy unsqueeze path for the safe default batch size of one.
    if len(batch) == 1:
        # Extract the only sample dictionary.
        item = batch[0]
        # Add a batch dimension without duplicating the large image tensor.
        result = {
            "study_uid": [item["study_uid"]],
            "volumes": item["volumes"].unsqueeze(0),
            "slice_position": item["slice_position"].unsqueeze(0),
            "present": item["present"].unsqueeze(0),
            "series_meta": item["series_meta"].unsqueeze(0),
        }
        # Add a label tensor only for a labelled training or validation sample.
        if "target" in item:
            result["target"] = item["target"].unsqueeze(0)
        # Return the memory-safe one-study batch.
        return result
    # Find the largest series count in this multi-study batch.
    max_series = max(int(item["present"].shape[0]) for item in batch)
    # Read image shape information from the first study.
    first = batch[0]["volumes"]
    # Read the actual batch size.
    size = len(batch)
    # Unpack one series tensor shape.
    _, slices, channels, height, width = first.shape
    # Allocate padded image storage.
    volumes = first.new_zeros((size, max_series, slices, channels, height, width))
    # Allocate padded position storage.
    positions = torch.zeros((size, max_series, slices), dtype=torch.float32)
    # Allocate padded readable-series flags.
    present = torch.zeros((size, max_series), dtype=torch.float32)
    # Allocate padded categorical metadata.
    metadata = torch.zeros((size, max_series, 3), dtype=torch.long)
    # Copy each study's variable number of series into the padded tensors.
    for row, item in enumerate(batch):
        # Read this study's real series count.
        count = int(item["present"].shape[0])
        # Copy image triplets.
        volumes[row, :count] = item["volumes"]
        # Copy through-plane positions.
        positions[row, :count] = item["slice_position"]
        # Copy readable-series flags.
        present[row, :count] = item["present"]
        # Copy categorical metadata.
        metadata[row, :count] = item["series_meta"]
    # Build a normal multi-study batch dictionary.
    result = {
        "study_uid": [item["study_uid"] for item in batch],
        "volumes": volumes,
        "slice_position": positions,
        "present": present,
        "series_meta": metadata,
    }
    # Stack labels only if every sample in the batch provides them.
    if all("target" in item for item in batch):
        result["target"] = torch.stack([item["target"] for item in batch])
    # Return the padded train or test batch.
    return result


def make_split(frame: pd.DataFrame, fraction: float, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Make a deterministic study-level train/validation split."""
    # Validate the requested validation fraction.
    if not 0 <= fraction < 1:
        raise ValueError("validation_fraction must be in [0, 1)")
    # Create a seeded random permutation of row positions.
    order = np.random.default_rng(seed).permutation(len(frame))
    # Keep one validation case when at least two studies exist.
    validation_size = 0 if len(frame) < 2 else max(1, int(round(len(frame) * fraction)))
    # Store selected validation positions in a set for quick membership checks.
    validation_indices = set(order[:validation_size].tolist())
    # Keep non-validation rows for training.
    train = frame.loc[[index not in validation_indices for index in range(len(frame))]].reset_index(drop=True)
    # Keep validation rows in a separate frame.
    validation = frame.loc[[index in validation_indices for index in range(len(frame))]].reset_index(drop=True)
    # Stop if a pathological fraction would remove every training row.
    if train.empty:
        raise ValueError("The split left no studies for training")
    # Return the two independent study tables.
    return train, validation

## 8. B48-shaped model: compact global pathology queries and conditioned sparse evidence

In [ ]:
class ConvNormAct(nn.Module):
    """A convolution, group normalization, and GELU activation block."""

    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__()
        groups = max(1, min(8, out_channels // 8))
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)


class ResidualBlock(nn.Module):
    """A compact residual block that preserves feature-map resolution."""

    def __init__(self, channels: int) -> None:
        super().__init__()
        groups = max(1, min(8, channels // 8))
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, channels),
            nn.GELU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, channels),
        )
        self.activation = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation(x + self.block(x))


class SliceEncoder(nn.Module):
    """Encode one 448×448 triplet into global and 6×6 local features."""

    def __init__(self, feature_dim: int, grid_size: int) -> None:
        super().__init__()
        self.feature_dim = int(feature_dim)
        self.grid_size = int(grid_size)
        self.stem = ConvNormAct(3, 32, stride=2)
        self.stage1 = nn.Sequential(ConvNormAct(32, 48, stride=2), ResidualBlock(48))
        self.stage2 = nn.Sequential(ConvNormAct(48, 72, stride=2), ResidualBlock(72))
        self.stage3 = nn.Sequential(ConvNormAct(72, 96, stride=2), ResidualBlock(96))
        self.stage4 = nn.Sequential(
            ConvNormAct(96, self.feature_dim, stride=2), ResidualBlock(self.feature_dim)
        )

    def forward(self, images: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        features = self.stage4(self.stage3(self.stage2(self.stage1(self.stem(images)))))
        global_feature = F.adaptive_avg_pool2d(features, 1).flatten(1)
        local_feature = F.adaptive_avg_pool2d(features, self.grid_size)
        return global_feature, local_feature


def position_basis(position: torch.Tensor) -> torch.Tensor:
    """Create the continuous eight-dimensional slice-position representation."""
    z = position.float().clamp(0.0, 1.0)
    return torch.stack(
        [
            z,
            z.square(),
            torch.sin(math.pi * z),
            torch.cos(math.pi * z),
            torch.sin(2 * math.pi * z),
            torch.cos(2 * math.pi * z),
            torch.sin(4 * math.pi * z),
            torch.cos(4 * math.pi * z),
        ],
        dim=-1,
    )


@dataclass
class B48SubsetHeadOutput:
    """Sparse local logits plus optional context/top-k audit values."""

    local_logits: torch.Tensor
    top_indices: torch.Tensor
    top_values: torch.Tensor
    context_abs_mean: torch.Tensor
    base_top_indices: torch.Tensor | None
    topk_overlap_with_static: torch.Tensor | None


@dataclass
class B48SubsetOutput:
    """Combined/global/local logits and the detached local-conditioning query."""

    logits: torch.Tensor
    global_logits: torch.Tensor
    local_logits: torch.Tensor
    context_query: torch.Tensor
    top_indices: torch.Tensor
    top_values: torch.Tensor
    context_abs_mean: torch.Tensor
    topk_overlap_with_static: torch.Tensor | None


class CompactGlobalPathologyBranch(nn.Module):
    """Compact B34-shaped series memory and pathology-query readout.

    This branch is newly initialized for the Drive subset.  It mirrors the B48
    *representation boundary*—static pathology prior versus a query after
    cross-attention over study-series memory—without claiming to be the frozen
    pretrained B34 hierarchy used by the full experiment.
    """

    def __init__(self, config: RunConfig) -> None:
        super().__init__()
        dim = int(config.feature_dim)
        heads = int(config.global_attention_heads)
        if dim % heads:
            raise ValueError("feature_dim must divide evenly across global_attention_heads")
        self.plane_embedding = nn.Embedding(4, dim, padding_idx=0)
        self.fluid_embedding = nn.Embedding(3, dim, padding_idx=0)
        self.fat_embedding = nn.Embedding(3, dim, padding_idx=0)
        self.series_norm = nn.LayerNorm(dim)
        layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=heads,
            dim_feedforward=2 * dim,
            dropout=float(config.global_dropout),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.context = nn.TransformerEncoder(layer, num_layers=int(config.global_memory_layers))
        self.pathology_tokens = nn.Parameter(torch.empty(N_TARGETS, dim))
        self.pathology_context = nn.Sequential(
            nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU()
        )
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=heads,
            dropout=float(config.global_dropout),
            batch_first=True,
        )
        self.query_norm = nn.LayerNorm(dim)
        self.target_weight = nn.Parameter(torch.empty(N_TARGETS, dim))
        self.target_bias = nn.Parameter(torch.zeros(N_TARGETS))
        nn.init.normal_(self.pathology_tokens, mean=0.0, std=0.02)
        nn.init.normal_(self.target_weight, mean=0.0, std=0.02)

    def forward(
        self,
        series_feature: torch.Tensor,
        present: torch.Tensor,
        series_meta: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Return prior queries, post-memory queries, and global logits."""
        if series_feature.ndim != 3:
            raise ValueError("global series features must be [B, S, D]")
        batch, series, _ = series_feature.shape
        if present.shape != (batch, series):
            raise ValueError("global present mask shape mismatch")
        metadata = (
            self.plane_embedding(series_meta[:, :, 0].clamp(0, 3))
            + self.fluid_embedding(series_meta[:, :, 1].clamp(0, 2))
            + self.fat_embedding(series_meta[:, :, 2].clamp(0, 2))
        ).to(series_feature.dtype)
        memory_input = self.series_norm(series_feature + metadata)
        memory_input = memory_input * present[:, :, None].to(memory_input.dtype)
        padding = present <= 0
        if bool(padding.all(dim=1).any()):
            raise RuntimeError("Each study needs at least one readable series")
        memory = self.context(memory_input, src_key_padding_mask=padding)
        memory = memory.masked_fill(padding[:, :, None], 0.0)
        raw_queries = self.pathology_tokens[None, :, :].expand(batch, -1, -1)
        prior = self.pathology_context(raw_queries)
        attended, _ = self.cross_attention(
            prior,
            memory,
            memory,
            key_padding_mask=padding,
            need_weights=False,
        )
        static_query = self.query_norm(prior)
        post_query = self.query_norm(prior + attended)
        global_logits = (
            post_query * self.target_weight[None, :, :]
        ).sum(dim=-1) + self.target_bias[None, :]
        return static_query, post_query, global_logits


class B48SubsetSparseEvidenceHead(nn.Module):
    """Top-k sparse MIL with B48's detached-query cosine residual."""

    def __init__(self, feature_dim: int, grid_size: int, top_k: int, context_dim: int = 96) -> None:
        super().__init__()
        self.feature_dim = int(feature_dim)
        self.grid_size = int(grid_size)
        self.top_k = int(top_k)
        self.context_dim = int(context_dim)
        self.n_regions = self.grid_size * self.grid_size
        if self.context_dim != 96:
            raise ValueError("This B48 subset notebook fixes context_dim=96")
        self.position_projection = nn.Linear(8, self.feature_dim, bias=False)
        self.region_embedding = nn.Parameter(torch.zeros(self.n_regions, self.feature_dim))
        self.plane_embedding = nn.Embedding(4, self.feature_dim, padding_idx=0)
        self.fluid_embedding = nn.Embedding(3, self.feature_dim, padding_idx=0)
        self.fat_embedding = nn.Embedding(3, self.feature_dim, padding_idx=0)
        self.evidence_weight = nn.Parameter(torch.empty(N_TARGETS, self.feature_dim))
        self.evidence_bias = nn.Parameter(torch.zeros(N_TARGETS))
        self.context_query = nn.Linear(self.feature_dim, self.context_dim, bias=False)
        self.context_key = nn.Linear(self.feature_dim, self.context_dim, bias=False)
        self.context_gate = nn.Parameter(torch.zeros(N_TARGETS))
        nn.init.normal_(self.evidence_weight, mean=0.0, std=0.02)
        nn.init.xavier_uniform_(self.context_query.weight)
        nn.init.xavier_uniform_(self.context_key.weight)

    def effective_context_gate(self) -> torch.Tensor:
        """Bound target-specific context contribution to [-1, 1]."""
        return torch.tanh(self.context_gate)

    def _tokens(
        self,
        spatial: torch.Tensor,
        present: torch.Tensor,
        series_meta: torch.Tensor,
        slice_position: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Add local coordinate/metadata context and construct the valid-token mask."""
        batch, series, slices, regions, feature_dim = spatial.shape
        if regions != self.n_regions or feature_dim != self.feature_dim:
            raise ValueError("Sparse feature shape does not match the B48 head")
        tokens = F.layer_norm(spatial.float(), (feature_dim,)).to(spatial.dtype)
        position = self.position_projection(position_basis(slice_position)).to(tokens.dtype)
        metadata = (
            self.plane_embedding(series_meta[:, :, 0].clamp(0, 3))
            + self.fluid_embedding(series_meta[:, :, 1].clamp(0, 2))
            + self.fat_embedding(series_meta[:, :, 2].clamp(0, 2))
        ).to(tokens.dtype)
        tokens = tokens + position[:, :, :, None, :]
        tokens = tokens + metadata[:, :, None, None, :]
        tokens = tokens + self.region_embedding.to(tokens.dtype)[None, None, None, :, :]
        tokens = tokens.reshape(batch, series * slices * regions, feature_dim)
        invalid = (
            (present <= 0)[:, :, None, None]
            .expand(batch, series, slices, regions)
            .reshape(batch, series * slices * regions)
        )
        if int((~invalid).sum(dim=1).min().item()) < self.top_k:
            raise RuntimeError("There are fewer valid local-MIL tokens than top_k")
        return tokens, invalid

    def _context_residual(
        self,
        tokens: torch.Tensor,
        global_query: torch.Tensor,
        invalid: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Compute B48's bounded detached-query compatibility residual."""
        batch, targets, feature_dim = global_query.shape
        if targets != N_TARGETS or feature_dim != self.feature_dim or batch != tokens.shape[0]:
            raise ValueError("B48 global-query shape does not match local tokens")
        # This is the explicit stop-gradient boundary required by B48.
        query = global_query.detach().float()
        query = F.layer_norm(query, (self.feature_dim,))
        token = F.layer_norm(tokens.float(), (self.feature_dim,))
        query = F.normalize(self.context_query(query), p=2.0, dim=-1, eps=1e-6)
        token = F.normalize(self.context_key(token), p=2.0, dim=-1, eps=1e-6)
        cosine = torch.einsum("btr,bnr->btn", query, token)
        residual = self.effective_context_gate().float()[None, :, None] * cosine
        valid = (~invalid).float()
        denominator = valid.sum(dim=-1).clamp_min(1.0)[:, None]
        context_abs_mean = (residual.abs() * valid[:, None, :]).sum(dim=-1) / denominator
        return residual, context_abs_mean

    def forward_details(
        self,
        spatial: torch.Tensor,
        present: torch.Tensor,
        series_meta: torch.Tensor,
        slice_position: torch.Tensor,
        global_query: torch.Tensor,
        *,
        audit_context: bool = False,
    ) -> B48SubsetHeadOutput:
        """Score all local tokens, rank top-k, and optionally audit rank changes."""
        tokens, invalid = self._tokens(spatial, present, series_meta, slice_position)
        base_score = torch.einsum(
            "bnd,td->btn", tokens, self.evidence_weight.to(tokens.dtype)
        ) + self.evidence_bias.to(tokens.dtype)[None, :, None]
        context_residual, context_abs_mean = self._context_residual(tokens, global_query, invalid)
        score = (base_score + context_residual.to(base_score.dtype)).masked_fill(
            invalid[:, None, :], float("-inf")
        )
        top_values, top_indices = torch.topk(score, k=self.top_k, dim=-1, largest=True, sorted=True)
        local_logits = torch.logsumexp(top_values.float(), dim=-1) - math.log(float(self.top_k))
        base_top_indices = overlap = None
        if audit_context:
            static_score = base_score.masked_fill(invalid[:, None, :], float("-inf"))
            base_top_indices = torch.topk(static_score, k=self.top_k, dim=-1, largest=True, sorted=True).indices
            overlap = (
                (top_indices[..., :, None] == base_top_indices[..., None, :])
                .any(dim=-1)
                .float()
                .mean(dim=-1)
            )
        return B48SubsetHeadOutput(
            local_logits=local_logits,
            top_indices=top_indices,
            top_values=top_values.float(),
            context_abs_mean=context_abs_mean.float(),
            base_top_indices=base_top_indices,
            topk_overlap_with_static=overlap,
        )


class B48SubsetModel(nn.Module):
    """Fresh compact model with the two B48 query-source arms."""

    ARMS = ("static_prior_control", "post_cross_attention_candidate")

    def __init__(self, config: RunConfig, arm: str) -> None:
        super().__init__()
        if arm not in self.ARMS:
            raise ValueError(f"arm must be one of {self.ARMS}; got {arm!r}")
        self.config = config
        self.arm = str(arm)
        self.context_source = (
            "pathology_prior_before_series_cross_attention"
            if arm == "static_prior_control"
            else "post_series_cross_attention_query"
        )
        self.encoder = SliceEncoder(config.feature_dim, config.grid_size)
        self.global_branch = CompactGlobalPathologyBranch(config)
        self.sparse_head = B48SubsetSparseEvidenceHead(
            config.feature_dim, config.grid_size, config.top_k, context_dim=config.context_dim
        )
        # As in B48, local logits enter the final prediction through a zero-start gate.
        self.fusion_gate = nn.Parameter(torch.zeros(N_TARGETS))

    def _encode_active_series(
        self, volumes: torch.Tensor, present: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Encode readable series only, then restore padded study shapes."""
        batch, series, slices, channels, height, width = volumes.shape
        if channels != 3:
            raise ValueError("The model expects three-channel 2.5D triplets")
        flat_series = volumes.reshape(batch * series, slices, channels, height, width)
        active_index = torch.nonzero(present.reshape(-1) > 0, as_tuple=False).flatten()
        if active_index.numel() == 0:
            raise RuntimeError("The batch has no readable MRI series")
        active = flat_series.index_select(0, active_index)
        images = active.reshape(-1, channels, height, width)
        global_blocks, local_blocks = [], []
        for image_chunk in images.split(self.config.encoder_chunk_size, dim=0):
            if self.training and self.config.gradient_checkpointing:
                global_feature, local_feature = checkpoint(self.encoder, image_chunk, use_reentrant=False)
            else:
                global_feature, local_feature = self.encoder(image_chunk)
            global_blocks.append(global_feature)
            local_blocks.append(local_feature)
        global_active = torch.cat(global_blocks, dim=0).reshape(
            active.shape[0], slices, self.config.feature_dim
        )
        local_active = torch.cat(local_blocks, dim=0).reshape(
            active.shape[0], slices, self.config.feature_dim, self.config.grid_size, self.config.grid_size
        )
        global_all = global_active.new_zeros((batch * series, slices, self.config.feature_dim))
        global_all.index_copy_(0, active_index, global_active)
        local_all = local_active.new_zeros(
            (batch * series, slices, self.config.feature_dim, self.config.grid_size, self.config.grid_size)
        )
        local_all.index_copy_(0, active_index, local_active)
        global_feature = global_all.reshape(batch, series, slices, self.config.feature_dim)
        spatial_feature = local_all.reshape(
            batch, series, slices, self.config.feature_dim, self.config.grid_size, self.config.grid_size
        ).permute(0, 1, 2, 4, 5, 3).reshape(
            batch, series, slices, self.config.grid_size * self.config.grid_size, self.config.feature_dim
        )
        return global_feature, spatial_feature

    def forward(
        self,
        volumes: torch.Tensor,
        present: torch.Tensor,
        series_meta: torch.Tensor,
        slice_position: torch.Tensor,
        *,
        audit_context: bool = False,
    ) -> B48SubsetOutput:
        """Return B48-shaped combined/global/local study predictions."""
        global_feature, spatial_feature = self._encode_active_series(volumes, present)
        series_feature = global_feature.mean(dim=2) * present[:, :, None].to(global_feature.dtype)
        static_query, post_query, global_logits = self.global_branch(
            series_feature, present, series_meta
        )
        # Select only the query source under comparison, then detach it before local scoring.
        context_query = (
            static_query if self.arm == "static_prior_control" else post_query
        ).detach()
        details = self.sparse_head.forward_details(
            spatial_feature,
            present,
            series_meta,
            slice_position,
            context_query,
            audit_context=audit_context,
        )
        logits = global_logits.float() + torch.tanh(self.fusion_gate)[None, :] * details.local_logits.float()
        return B48SubsetOutput(
            logits=logits,
            global_logits=global_logits,
            local_logits=details.local_logits,
            context_query=context_query,
            top_indices=details.top_indices,
            top_values=details.top_values,
            context_abs_mean=details.context_abs_mean,
            topk_overlap_with_static=details.topk_overlap_with_static,
        )

## 9. Paired-arm loss, preflight, training, and comparison functions

In [ ]:
def masked_bce_with_logits(
    logits: torch.Tensor,
    target: torch.Tensor,
    positive_weight: torch.Tensor,
) -> torch.Tensor:
    """Compute weighted BCE only for non-blank subset target cells."""
    known = torch.isfinite(target)
    if not bool(known.any()):
        raise RuntimeError("This batch has no usable supervision cells")
    safe_target = torch.nan_to_num(target, nan=0.0)
    loss = F.binary_cross_entropy_with_logits(
        logits.float(), safe_target.float(), pos_weight=positive_weight.float(), reduction="none"
    )
    return (loss * known).sum() / known.sum().clamp_min(1)


def make_positive_weight(frame: pd.DataFrame) -> torch.Tensor:
    """Build clipped target-wise positive weights from the common train rows."""
    labels = frame[TARGETS].apply(pd.to_numeric, errors="coerce")
    known = labels.notna().sum(axis=0).to_numpy(np.float32)
    positive = labels.fillna(0).sum(axis=0).to_numpy(np.float32)
    negative = np.maximum(known - positive, 1.0)
    weight = np.clip(negative / np.maximum(positive, 1.0), 1.0, 20.0)
    return torch.tensor(weight, dtype=torch.float32, device=DEVICE)


def binary_auc(target: np.ndarray, probability: np.ndarray) -> float | None:
    """Compute ROC-AUC without an additional dependency; return None for one class."""
    target = np.asarray(target, dtype=np.int64)
    probability = np.asarray(probability, dtype=np.float64)
    positive = int(target.sum())
    negative = int(len(target) - positive)
    if positive == 0 or negative == 0:
        return None
    order = np.argsort(probability, kind="mergesort")
    ranks = np.empty(len(probability), dtype=np.float64)
    ranks[order] = np.arange(1, len(probability) + 1, dtype=np.float64)
    ordered_probability = probability[order]
    start = 0
    while start < len(ordered_probability):
        end = start + 1
        while end < len(ordered_probability) and ordered_probability[end] == ordered_probability[start]:
            end += 1
        if end - start > 1:
            ranks[order[start:end]] = ranks[order[start:end]].mean()
        start = end
    return float((ranks[target == 1].sum() - positive * (positive + 1) / 2) / (positive * negative))


def evaluate_predictions(target: np.ndarray, probability: np.ndarray) -> dict:
    """Calculate defined per-target and macro AUC values for the subset split."""
    per_target: dict[str, float | None] = {}
    for index, name in enumerate(TARGETS):
        known = np.isfinite(target[:, index])
        per_target[name] = (
            binary_auc(target[known, index].astype(int), probability[known, index])
            if known.any() else None
        )
    defined = [value for value in per_target.values() if value is not None]
    return {
        "mean_auc": None if not defined else float(np.mean(defined)),
        "per_target_auc": per_target,
        "known_cells": int(np.isfinite(target).sum()),
    }


def move_model_inputs(batch: dict) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Move only B48 model inputs to GPU/CPU without pinned-memory pressure."""
    return (
        batch["volumes"].to(DEVICE, non_blocking=False),
        batch["present"].to(DEVICE, non_blocking=False),
        batch["series_meta"].to(DEVICE, non_blocking=False),
        batch["slice_position"].to(DEVICE, non_blocking=False),
    )


def move_batch(batch: dict) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Move one labelled batch in the model-forward/loss order."""
    volumes, present, metadata, position = move_model_inputs(batch)
    return volumes, present, metadata, position, batch["target"].to(DEVICE, non_blocking=False)


def autocast_context():
    """Use CUDA fp16 autocast when available and ordinary precision otherwise."""
    return torch.autocast(device_type="cuda", dtype=torch.float16) if DEVICE.type == "cuda" else nullcontext()


@dataclass
class B48SubsetExperiment:
    """All run objects for one arm of the matched Drive-subset comparison."""

    arm: str
    paths: DrivePaths
    config: RunConfig
    model: B48SubsetModel
    optimizer: torch.optim.Optimizer
    scaler: object
    train_loader: DataLoader
    validation_loader: DataLoader | None
    positive_weight: torch.Tensor
    train_uid_sha256: str
    validation_uid_sha256: str
    history: list[dict] = field(default_factory=list)


@dataclass
class B48MatchedPair:
    """The two B48 query-source arms sharing one split and initialization."""

    static_prior_control: B48SubsetExperiment
    post_cross_attention_candidate: B48SubsetExperiment
    initialization_fingerprint: str

    def arms(self) -> tuple[B48SubsetExperiment, B48SubsetExperiment]:
        return self.static_prior_control, self.post_cross_attention_candidate


def _uid_sha256(frame: pd.DataFrame) -> str:
    """Fingerprint the exact study membership of a split without exposing data."""
    payload = "\n".join(frame["StudyInstanceUID"].astype(str).tolist()) + "\n"
    return __import__("hashlib").sha256(payload.encode("utf-8")).hexdigest()


def _model_fingerprint(model: nn.Module) -> str:
    """Fingerprint all initialized tensors to prove the two arm states match."""
    digest = __import__("hashlib").sha256()
    for name, tensor in model.state_dict().items():
        digest.update(name.encode("utf-8"))
        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def _make_loader(dataset: Dataset, config: RunConfig, *, shuffle: bool, seed: int) -> DataLoader:
    """Build an arm-private loader whose shuffled study order is reproducible."""
    generator = torch.Generator()
    generator.manual_seed(int(seed))
    return DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=shuffle,
        generator=generator,
        num_workers=config.num_workers,
        pin_memory=False,
        collate_fn=collate_studies,
    )


def _construct_matched_model(config: RunConfig, arm: str) -> B48SubsetModel:
    """Construct either arm under the same private RNG stream."""
    devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(int(config.seed) + 481516)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(int(config.seed) + 481516)
        model = B48SubsetModel(config, arm)
    return model.to(DEVICE)


def _prepare_subset_split(paths: DrivePaths, config: RunConfig) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, list[dict]]]:
    """Read the shared subset once and make the common deterministic split."""
    validate_dataset(paths)
    train_table = pd.read_csv(paths.train_csv)
    series_table = pd.read_csv(paths.series_csv)
    train_table["StudyInstanceUID"] = train_table["StudyInstanceUID"].astype(str)
    records = build_series_records(series_table, config)
    labels = train_table[TARGETS].apply(pd.to_numeric, errors="coerce")
    usable = train_table["StudyInstanceUID"].isin(records) & labels.notna().any(axis=1)
    usable_table = train_table.loc[usable].reset_index(drop=True)
    if usable_table.empty:
        raise ValueError("No studies remain after matching labels to readable MRI metadata")
    train_frame, validation_frame = make_split(
        usable_table, config.validation_fraction, config.seed
    )
    return train_frame, validation_frame, records


def build_b48_matched_pair(paths: DrivePaths, config: RunConfig = CONFIG) -> B48MatchedPair:
    """Build both arms from exactly the same subset split and initial state."""
    set_seed(config.seed)
    train_frame, validation_frame, records = _prepare_subset_split(paths, config)
    train_dataset = KneeMRIDataset(
        train_frame, records, paths, config, split="train", include_targets=True
    )
    validation_dataset = (
        KneeMRIDataset(validation_frame, records, paths, config, split="train", include_targets=True)
        if not validation_frame.empty else None
    )
    positive_weight = make_positive_weight(train_frame)
    train_uid_sha256, validation_uid_sha256 = _uid_sha256(train_frame), _uid_sha256(validation_frame)

    def make_arm(arm: str) -> B48SubsetExperiment:
        model = _construct_matched_model(config, arm)
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
        )
        return B48SubsetExperiment(
            arm=arm,
            paths=paths,
            config=config,
            model=model,
            optimizer=optimizer,
            scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda"),
            train_loader=_make_loader(train_dataset, config, shuffle=True, seed=config.seed + 91),
            validation_loader=(
                _make_loader(validation_dataset, config, shuffle=False, seed=config.seed + 92)
                if validation_dataset is not None else None
            ),
            positive_weight=positive_weight,
            train_uid_sha256=train_uid_sha256,
            validation_uid_sha256=validation_uid_sha256,
        )

    control = make_arm("static_prior_control")
    candidate = make_arm("post_cross_attention_candidate")
    control_fingerprint = _model_fingerprint(control.model)
    candidate_fingerprint = _model_fingerprint(candidate.model)
    if control_fingerprint != candidate_fingerprint:
        raise RuntimeError("Matched B48 arms did not start from identical parameter tensors")
    print(
        f"train studies={len(train_dataset)} | validation studies={0 if validation_dataset is None else len(validation_dataset)} | "
        f"matched_initialization={control_fingerprint[:16]}"
    )
    return B48MatchedPair(control, candidate, control_fingerprint)


def _has_nonzero_gradient(parameters: Iterable[nn.Parameter]) -> bool:
    """Return whether any supplied parameter has a nonzero gradient."""
    return any(
        parameter.grad is not None and bool(torch.count_nonzero(parameter.grad).item())
        for parameter in parameters
    )


def check_zero_start_pair_equivalence(pair: B48MatchedPair) -> float:
    """Verify that the two arms are numerically identical while both gates are closed.

    The probe is synthetic and small, so it does not consume a shuffled DICOM
    batch or disturb either arm's future train-loader order.  At initialization
    the models have identical tensors, the local context gate is zero, and the
    final local-fusion gate is zero.  Therefore static and post-attention query
    selection must have no effect on any output logit yet.
    """
    control, candidate = pair.arms()
    was_training = (control.model.training, candidate.model.training)
    control.model.eval()
    candidate.model.eval()
    probe_side, probe_slices = 192, 2
    volumes = torch.zeros((1, 1, probe_slices, 3, probe_side, probe_side), device=DEVICE)
    present = torch.ones((1, 1), dtype=torch.float32, device=DEVICE)
    metadata = torch.zeros((1, 1, 3), dtype=torch.long, device=DEVICE)
    position = torch.linspace(0.0, 1.0, probe_slices, device=DEVICE)[None, None, :]
    with torch.no_grad(), autocast_context():
        control_output = control.model(volumes, present, metadata, position)
        candidate_output = candidate.model(volumes, present, metadata, position)
    max_abs = max(
        float((control_output.logits.float() - candidate_output.logits.float()).abs().max().cpu()),
        float((control_output.global_logits.float() - candidate_output.global_logits.float()).abs().max().cpu()),
        float((control_output.local_logits.float() - candidate_output.local_logits.float()).abs().max().cpu()),
    )
    control.model.train(was_training[0])
    candidate.model.train(was_training[1])
    del volumes, present, metadata, position, control_output, candidate_output
    if max_abs > 1e-5:
        raise RuntimeError(f"B48 arms differ before a zero-start gate can open: max_abs={max_abs:.3e}")
    return max_abs


def run_b48_preflight(experiment: B48SubsetExperiment) -> dict:
    """Test B48's gradient boundary without a single optimizer step."""
    print(f"[{experiment.arm}] preflight: forward/backward only; no optimizer step")
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    model = experiment.model
    model.train()
    batch = next(iter(experiment.train_loader))
    volumes, present, metadata, position, target = move_batch(batch)
    del batch
    saved_context_gate = model.sparse_head.context_gate.detach().clone()

    def local_loss_only() -> tuple[B48SubsetOutput, torch.Tensor]:
        with autocast_context():
            output = model(volumes, present, metadata, position, audit_context=True)
            loss = masked_bce_with_logits(output.local_logits, target, experiment.positive_weight)
        return output, loss

    model.zero_grad(set_to_none=True)
    with autocast_context():
        output = model(volumes, present, metadata, position, audit_context=True)
        combined_loss = masked_bce_with_logits(output.logits, target, experiment.positive_weight)
        local_loss = masked_bce_with_logits(output.local_logits, target, experiment.positive_weight)
        total_loss = combined_loss + experiment.config.local_loss_weight * local_loss
    experiment.scaler.scale(total_loss).backward()
    encoder_gradient = _has_nonzero_gradient(model.encoder.parameters())
    sparse_gradient = _has_nonzero_gradient([model.sparse_head.evidence_weight])
    if not encoder_gradient or not sparse_gradient:
        raise RuntimeError("Preflight failed: encoder and sparse evidence need gradients")
    model.zero_grad(set_to_none=True)

    # Local-only supervision may update spatial features and B48 gates, but not the global query branch.
    detached_output, detached_loss = local_loss_only()
    if detached_output.context_query.requires_grad:
        raise RuntimeError("B48 context query was not detached before local conditioning")
    experiment.scaler.scale(detached_loss).backward()
    gate_gradient = _has_nonzero_gradient([model.sparse_head.context_gate])
    leaked_global_gradient = any(parameter.grad is not None for parameter in model.global_branch.parameters())
    projections_still_closed = all(
        parameter.grad is None or not bool(torch.count_nonzero(parameter.grad).item())
        for parameter in (model.sparse_head.context_query.weight, model.sparse_head.context_key.weight)
    )
    if not gate_gradient or leaked_global_gradient or not projections_still_closed:
        raise RuntimeError("B48 zero-start detached-query gradient contract failed")
    model.zero_grad(set_to_none=True)

    # Opening the gate synthetically verifies that both low-rank projections become trainable later.
    with torch.no_grad():
        model.sparse_head.context_gate.fill_(0.05)
    opened_output, opened_loss = local_loss_only()
    experiment.scaler.scale(opened_loss).backward()
    projections_open = _has_nonzero_gradient(
        [model.sparse_head.context_query.weight, model.sparse_head.context_key.weight]
    )
    leaked_after_open = any(parameter.grad is not None for parameter in model.global_branch.parameters())
    with torch.no_grad():
        model.sparse_head.context_gate.copy_(saved_context_gate)
    model.zero_grad(set_to_none=True)
    if not projections_open or leaked_after_open:
        raise RuntimeError("B48 opened-gate or detachment contract failed")

    result = {
        "arm": experiment.arm,
        "total_loss": float(total_loss.detach().cpu()),
        "combined_loss": float(combined_loss.detach().cpu()),
        "local_loss": float(local_loss.detach().cpu()),
        "encoder_gradient": bool(encoder_gradient),
        "sparse_evidence_gradient": bool(sparse_gradient),
        "context_gate_gradient_at_zero": bool(gate_gradient),
        "context_projections_zero_at_zero_gate": bool(projections_still_closed),
        "context_projections_active_after_opening": bool(projections_open),
        "global_branch_isolated_from_local_loss": not leaked_global_gradient and not leaked_after_open,
        "host_peak_rss_gib": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024**2, 2),
        "input_batch_gib": round(volumes.numel() * volumes.element_size() / 1024**3, 2),
    }
    if DEVICE.type == "cuda":
        result["cuda_peak_gib"] = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
    del volumes, present, metadata, position, target, output, combined_loss, local_loss, total_loss
    del detached_output, detached_loss, opened_output, opened_loss
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print(json.dumps(result, indent=2))
    print(f"[{experiment.arm}] preflight: PASS")
    return result


def run_b48_pair_preflight(pair: B48MatchedPair) -> dict:
    """Run the no-update B48 check separately for both matched arms."""
    control, candidate = pair.arms()
    if control.train_uid_sha256 != candidate.train_uid_sha256:
        raise RuntimeError("Matched arms have different training split membership")
    if control.validation_uid_sha256 != candidate.validation_uid_sha256:
        raise RuntimeError("Matched arms have different validation split membership")
    if _model_fingerprint(control.model) != pair.initialization_fingerprint:
        raise RuntimeError("Control initialization changed before preflight")
    if _model_fingerprint(candidate.model) != pair.initialization_fingerprint:
        raise RuntimeError("Candidate initialization changed before preflight")
    zero_start_max_abs = check_zero_start_pair_equivalence(pair)
    return {
        "matched_initialization_fingerprint": pair.initialization_fingerprint,
        "zero_start_pair_max_abs_difference": zero_start_max_abs,
        "static_prior_control": run_b48_preflight(control),
        "post_cross_attention_candidate": run_b48_preflight(candidate),
    }


def run_b48_epoch(experiment: B48SubsetExperiment, loader: DataLoader, training: bool) -> dict:
    """Run one train/validation pass and preserve B48 context/top-k audit arrays."""
    experiment.model.train(training)
    losses, targets, probabilities, context_abs, overlaps = [], [], [], [], []
    for batch in loader:
        volumes, present, metadata, position, target = move_batch(batch)
        del batch
        if training:
            experiment.optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training), autocast_context():
            output = experiment.model(
                volumes, present, metadata, position, audit_context=not training
            )
            combined_loss = masked_bce_with_logits(output.logits, target, experiment.positive_weight)
            local_loss = masked_bce_with_logits(output.local_logits, target, experiment.positive_weight)
            loss = combined_loss + experiment.config.local_loss_weight * local_loss
        if training:
            experiment.scaler.scale(loss).backward()
            experiment.scaler.unscale_(experiment.optimizer)
            torch.nn.utils.clip_grad_norm_(experiment.model.parameters(), experiment.config.grad_clip_norm)
            experiment.scaler.step(experiment.optimizer)
            experiment.scaler.update()
        losses.append(float(loss.detach().cpu()))
        targets.append(target.detach().cpu().numpy())
        probabilities.append(torch.sigmoid(output.logits).detach().cpu().numpy())
        context_abs.append(output.context_abs_mean.detach().cpu().numpy())
        if output.topk_overlap_with_static is not None:
            overlaps.append(output.topk_overlap_with_static.detach().cpu().numpy())
        del volumes, present, metadata, position, target, output, loss, combined_loss, local_loss
    return {
        "loss": float(np.mean(losses)),
        "target": np.concatenate(targets, axis=0),
        "probability": np.concatenate(probabilities, axis=0),
        "context_abs_mean": np.concatenate(context_abs, axis=0),
        "topk_overlap_with_static": None if not overlaps else np.concatenate(overlaps, axis=0),
    }


def train_b48_arm(experiment: B48SubsetExperiment) -> list[dict]:
    """Train one arm for the fixed subset duration using the common RNG seed."""
    # Reset stochastic layers for each arm; each private loader already has the same order seed.
    set_seed(experiment.config.seed + 97531)
    for epoch in range(1, experiment.config.epochs + 1):
        started = time.time()
        train_result = run_b48_epoch(experiment, experiment.train_loader, training=True)
        row = {"arm": experiment.arm, "epoch": epoch, "train_loss": train_result["loss"]}
        if experiment.validation_loader is not None:
            validation_result = run_b48_epoch(experiment, experiment.validation_loader, training=False)
            metrics = evaluate_predictions(validation_result["target"], validation_result["probability"])
            row.update(
                {
                    "validation_loss": validation_result["loss"],
                    "validation_mean_auc": metrics["mean_auc"],
                    "validation_known_cells": metrics["known_cells"],
                    "validation_context_abs_mean": float(validation_result["context_abs_mean"].mean()),
                    "validation_topk_change_fraction": (
                        None if validation_result["topk_overlap_with_static"] is None
                        else float(1.0 - validation_result["topk_overlap_with_static"].mean())
                    ),
                }
            )
        row["context_gate_abs_mean"] = float(
            experiment.model.sparse_head.effective_context_gate().detach().abs().mean().cpu()
        )
        row["fusion_gate_abs_mean"] = float(torch.tanh(experiment.model.fusion_gate).detach().abs().mean().cpu())
        row["elapsed_seconds"] = round(time.time() - started, 1)
        experiment.history.append(row)
        print(json.dumps(row, indent=2))
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return experiment.history


def train_b48_matched_pair(pair: B48MatchedPair) -> dict:
    """Train control first and candidate second; neither arm selects the other."""
    control, candidate = pair.arms()
    return {
        "static_prior_control": train_b48_arm(control),
        "post_cross_attention_candidate": train_b48_arm(candidate),
    }


def _evaluation_payload(experiment: B48SubsetExperiment) -> dict:
    """Run one fresh validation evaluation and serialize the arm's audit surface."""
    if experiment.validation_loader is None:
        raise ValueError("A validation split is needed for paired subset comparison")
    result = run_b48_epoch(experiment, experiment.validation_loader, training=False)
    metrics = evaluate_predictions(result["target"], result["probability"])
    per_target_context = dict(zip(TARGETS, result["context_abs_mean"].mean(axis=0).astype(float).tolist()))
    per_target_change = None
    if result["topk_overlap_with_static"] is not None:
        changes = 1.0 - result["topk_overlap_with_static"].mean(axis=0)
        per_target_change = dict(zip(TARGETS, changes.astype(float).tolist()))
    return {
        "validation_weighted_bce": result["loss"],
        "validation_metrics": metrics,
        "context_abs_mean_by_target": per_target_context,
        "topk_change_fraction_by_target": per_target_change,
        "effective_context_gate": dict(
            zip(
                TARGETS,
                experiment.model.sparse_head.effective_context_gate().detach().cpu().float().tolist(),
            )
        ),
        "effective_fusion_gate": dict(
            zip(TARGETS, torch.tanh(experiment.model.fusion_gate).detach().cpu().float().tolist())
        ),
    }


def evaluate_b48_matched_pair(pair: B48MatchedPair) -> dict:
    """Create the non-selective paired subset comparison and top-k change audit."""
    control, candidate = pair.arms()
    control_payload = _evaluation_payload(control)
    candidate_payload = _evaluation_payload(candidate)
    control_auc = control_payload["validation_metrics"]["per_target_auc"]
    candidate_auc = candidate_payload["validation_metrics"]["per_target_auc"]
    per_target_delta = {
        target: (
            None if control_auc[target] is None or candidate_auc[target] is None
            else float(candidate_auc[target] - control_auc[target])
        )
        for target in TARGETS
    }
    return {
        "scope": asdict(B48_SUBSET_REFERENCE),
        "matched_initialization_fingerprint": pair.initialization_fingerprint,
        "common_train_uid_sha256": control.train_uid_sha256,
        "common_validation_uid_sha256": control.validation_uid_sha256,
        "fixed_epochs": int(control.config.epochs),
        "static_prior_control": control_payload,
        "post_cross_attention_candidate": candidate_payload,
        "candidate_minus_control": {
            "validation_mean_auc": (
                None
                if control_payload["validation_metrics"]["mean_auc"] is None
                or candidate_payload["validation_metrics"]["mean_auc"] is None
                else float(
                    candidate_payload["validation_metrics"]["mean_auc"]
                    - control_payload["validation_metrics"]["mean_auc"]
                )
            ),
            "validation_weighted_bce": float(
                candidate_payload["validation_weighted_bce"]
                - control_payload["validation_weighted_bce"]
            ),
            "per_target_auc": per_target_delta,
        },
        "interpretation": (
            "Subset-only paired diagnostic; do not treat this as the official B48 scanner-domain result "
            "or use it for checkpoint/architecture selection."
        ),
    }


def plot_b48_pair_history(pair: B48MatchedPair) -> None:
    """Plot the matched arms' training and validation losses without selecting one."""
    plt.figure(figsize=(9, 4))
    for experiment in pair.arms():
        history = pd.DataFrame(experiment.history)
        if history.empty:
            raise ValueError("No completed epochs yet; train both B48 arms first")
        plt.plot(history["epoch"], history["train_loss"], marker="o", label=f"{experiment.arm} train")
        if "validation_loss" in history:
            plt.plot(
                history["epoch"],
                history["validation_loss"],
                marker="x",
                linestyle="--",
                label=f"{experiment.arm} validation",
            )
    plt.xlabel("epoch")
    plt.ylabel("masked weighted BCE loss")
    plt.title("B48-shaped matched subset arms (descriptive only)")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def show_b48_pair_results(pair: B48MatchedPair, comparison: dict) -> None:
    """Display both histories and the compact paired comparison summary."""
    history = pd.concat([pd.DataFrame(experiment.history) for experiment in pair.arms()], ignore_index=True)
    display(history)
    summary = pd.DataFrame(
        [
            {
                "arm": arm,
                "validation_mean_auc": comparison[arm]["validation_metrics"]["mean_auc"],
                "validation_weighted_bce": comparison[arm]["validation_weighted_bce"],
                "topk_change_fraction_mean": None
                if comparison[arm]["topk_change_fraction_by_target"] is None
                else float(np.mean(list(comparison[arm]["topk_change_fraction_by_target"].values()))),
            }
            for arm in ("static_prior_control", "post_cross_attention_candidate")
        ]
    )
    display(summary)


def build_test_loader(paths: TestPaths, config: RunConfig = CONFIG) -> DataLoader:
    """Build a no-label test loader from the same extracted Drive subset."""
    validate_test_dataset(paths)
    test_table = pd.read_csv(paths.test_csv)
    series_table = pd.read_csv(paths.series_csv)
    test_table["StudyInstanceUID"] = test_table["StudyInstanceUID"].astype(str)
    records = build_series_records(series_table, config)
    test_table = test_table.loc[test_table["StudyInstanceUID"].isin(records)].reset_index(drop=True)
    if test_table.empty:
        raise ValueError("No test studies remain after matching metadata")
    dataset = KneeMRIDataset(test_table, records, paths, config, split="test", include_targets=False)
    print(f"test studies={len(dataset)}")
    return _make_loader(dataset, config, shuffle=False, seed=config.seed + 93)


def predict_test_set(experiment: B48SubsetExperiment, test_loader: DataLoader) -> pd.DataFrame:
    """Generate test probabilities for one arm without updating its model."""
    experiment.model.eval()
    fragments: list[pd.DataFrame] = []
    with torch.no_grad():
        for batch in test_loader:
            study_uids = list(batch["study_uid"])
            volumes, present, metadata, position = move_model_inputs(batch)
            with autocast_context():
                output = experiment.model(volumes, present, metadata, position)
            probability = torch.sigmoid(output.logits).float().cpu().numpy()
            fragment = pd.DataFrame(probability, columns=TARGETS)
            fragment.insert(0, "StudyInstanceUID", study_uids)
            fragment["predicted_positive"] = [format_positive_predictions(row) for row in probability]
            fragments.append(fragment)
            del batch, volumes, present, metadata, position, output
    predictions = pd.concat(fragments, ignore_index=True)
    print(f"[{experiment.arm}] generated predictions for {len(predictions)} test studies")
    return predictions


def save_b48_pair_results(
    pair: B48MatchedPair,
    comparison: dict,
    test_predictions: dict[str, pd.DataFrame] | None = None,
) -> Path:
    """Save both models, histories, comparison/audit, and optional test predictions to Drive."""
    run_name = time.strftime("b48_subset_matched_pair_%Y%m%d_%H%M%S")
    run_root = pair.static_prior_control.paths.output_root / run_name
    run_root.mkdir(parents=True, exist_ok=False)
    for experiment in pair.arms():
        arm = experiment.arm
        torch.save(
            {
                "model_state": experiment.model.state_dict(),
                "config": asdict(experiment.config),
                "targets": TARGETS,
                "arm": arm,
                "context_source": experiment.model.context_source,
                "matched_initialization_fingerprint": pair.initialization_fingerprint,
                "b48_subset_scope": asdict(B48_SUBSET_REFERENCE),
            },
            run_root / f"{arm}_model.pt",
        )
        (run_root / f"{arm}_history.json").write_text(
            json.dumps(experiment.history, indent=2), encoding="utf-8"
        )
    (run_root / "config.json").write_text(
        json.dumps(asdict(pair.static_prior_control.config), indent=2), encoding="utf-8"
    )
    (run_root / "b48_subset_comparison.json").write_text(
        json.dumps(comparison, indent=2), encoding="utf-8"
    )
    (run_root / "b48_subset_scope.json").write_text(
        json.dumps(asdict(B48_SUBSET_REFERENCE), indent=2), encoding="utf-8"
    )
    if test_predictions is not None:
        for arm, prediction in test_predictions.items():
            prediction.to_csv(run_root / f"{arm}_test_predictions.csv", index=False)
    print("Saved matched B48 subset run to:", run_root)
    return run_root


# Retain the generic name used by the inherited case-review helper below.
Experiment = B48SubsetExperiment

## 10. Twelve-case classification review

In [ ]:
def format_positive_predictions(probability: np.ndarray, threshold: float = 0.50) -> str:
    """Format all target probabilities at or above the classification threshold."""
    # Build short target/probability strings for positive classifications.
    selected = [
        f"{name}: {value:.2f}"
        for name, value in zip(TARGETS, probability)
        if value >= threshold
    ]
    # Return a clear sentence when no target reaches the threshold.
    return "none at or above 0.50" if not selected else "; ".join(selected)


def format_known_labels(target: np.ndarray | None) -> str:
    """Format known labels when available, or clearly mark an unlabelled test case."""
    # Explain why the test subset cannot show truth labels.
    if target is None:
        return "test labels unavailable"
    # Build readable target/value pairs while skipping blank target cells.
    selected = [
        f"{name}: {int(value)}"
        for name, value in zip(TARGETS, target)
        if np.isfinite(value)
    ]
    # Return a clear fallback if a case unexpectedly has no known target cell.
    return "no known labels" if not selected else "; ".join(selected)


def collect_case_examples(experiment: Experiment, loader: DataLoader, max_cases: int) -> list[dict]:
    """Collect up to max_cases images and classifications from labelled or unlabelled data."""
    # Put the model in evaluation mode so dropout is disabled.
    experiment.model.eval()
    # Prepare an output list of per-study review records.
    cases: list[dict] = []
    # Disable gradients because case review performs inference only.
    with torch.no_grad():
        # Iterate through validation or training batches.
        for batch in loader:
            # Keep a CPU copy of study IDs before moving tensors to the device.
            study_uids = list(batch["study_uid"])
            # Keep CPU tensors for image display.
            cpu_volumes = batch["volumes"]
            # Keep CPU readable-series flags to select a real MRI image.
            cpu_present = batch["present"]
            # Keep CPU targets when this loader represents a labelled train or validation split.
            cpu_targets = batch.get("target")
            # Move model inputs to the selected device.
            volumes, present, metadata, position = move_model_inputs(batch)
            # Run a combined-logit inference pass.
            with autocast_context():
                output = experiment.model(volumes, present, metadata, position)
            # Convert logits to CPU probabilities.
            probabilities = torch.sigmoid(output.logits).float().cpu().numpy()
            # Add one visual review record per study in this batch.
            for row, study_uid in enumerate(study_uids):
                # Find the first readable MRI series for this study.
                first_series = int(torch.nonzero(cpu_present[row] > 0, as_tuple=False)[0].item())
                # Select the central sampled triplet and its middle channel for display.
                image = cpu_volumes[row, first_series, experiment.config.slices_per_series // 2, 1].numpy()
                # Copy targets into a NumPy row only when labels are present.
                target = None if cpu_targets is None else cpu_targets[row].numpy().copy()
                # Copy predicted probabilities into a NumPy row.
                probability = probabilities[row].copy()
                # Save every element needed for plotting and the summary table.
                cases.append(
                    {
                        "StudyInstanceUID": study_uid,
                        "image": image,
                        "target": target,
                        "probability": probability,
                        "known_labels": format_known_labels(target),
                        "predicted_positive": format_positive_predictions(probability),
                    }
                )
                # Stop as soon as the requested case count is reached.
                if len(cases) >= max_cases:
                    return cases
            # Release this batch's large GPU objects before the next DICOM batch.
            del volumes, present, metadata, position, output
    # Return all cases if the loader had fewer than max_cases studies.
    return cases


def show_case_examples(
    experiment: Experiment,
    loader: DataLoader | None = None,
    max_cases: int = 12,
    title_prefix: str = "Case",
) -> pd.DataFrame:
    """Plot up to 12 labelled or test MRI examples with thresholded classifications."""
    # Prefer validation examples when no explicit loader was supplied.
    loader = loader or experiment.validation_loader or experiment.train_loader
    # Collect the requested examples and their predictions.
    cases = collect_case_examples(experiment, loader, max_cases)
    # Stop clearly if the selected loader contained no cases.
    if not cases:
        raise ValueError("No cases available for visualization")
    # Use three columns and enough rows for up to twelve images.
    columns = 3
    # Compute the number of required figure rows.
    rows = math.ceil(len(cases) / columns)
    # Create a spacious grid for MRI images and text annotations.
    figure, axes = plt.subplots(rows, columns, figsize=(18, 5.5 * rows))
    # Flatten axes so indexing also works when the grid has a single row.
    axes = np.asarray(axes).reshape(-1)
    # Draw every requested case.
    for axis, case in zip(axes, cases):
        # Render the central 2.5D middle-channel MRI slice in grayscale.
        axis.imshow(case["image"], cmap="gray")
        # Hide axes because pixel coordinates are not part of the case review.
        axis.axis("off")
        # Show the split label and study ID above the MRI image.
        axis.set_title(f"{title_prefix} study {case['StudyInstanceUID']}", fontsize=10)
        # Add known truth and thresholded classification below the image.
        axis.text(
            0.0,
            -0.08,
            "known: " + case["known_labels"] + "\n" + "predicted: " + case["predicted_positive"],
            transform=axis.transAxes,
            fontsize=8,
            va="top",
            wrap=True,
        )
    # Hide unused panels when fewer than twelve studies are available.
    for axis in axes[len(cases) :]:
        axis.axis("off")
    # Leave room for the classification text below each image.
    plt.tight_layout()
    # Display the twelve-case review figure.
    plt.show()
    # Build a concise tabular summary without embedding large image arrays.
    table = pd.DataFrame(
        {
            "StudyInstanceUID": [case["StudyInstanceUID"] for case in cases],
            "known_labels": [case["known_labels"] for case in cases],
            "predicted_positive": [case["predicted_positive"] for case in cases],
            "max_probability": [float(case["probability"].max()) for case in cases],
        }
    )
    # Display the table below the figure in Colab.
    display(table)
    # Return the table so it can be saved or filtered in another cell.
    return table


def show_results(experiment: Experiment) -> pd.DataFrame:
    """Display the numeric epoch history as a table."""
    # Convert completed epoch dictionaries into a DataFrame.
    table = pd.DataFrame(experiment.history)
    # Render the table in Colab.
    display(table)
    # Return the table for optional user analysis.
    return table

## 11. Build the matched B48 subset pair from the extracted Drive data

In [ ]:
# Build two fresh arms from the same CSV split and byte-identical initial parameter tensors.
B48_PAIR = build_b48_matched_pair(PATHS, CONFIG)

### 11a. Mandatory no-update B48 detachment, zero-gate, and memory check

In [ ]:
# Both arms must pass before either optimizer is allowed to take a step.
B48_PREFLIGHT = run_b48_pair_preflight(B48_PAIR)

### 11b. Train both fixed-duration arms, compare descriptively, predict, and save

In [ ]:
# Keep both optimizers off until the preflight cell reports PASS for both arms.
RUN_B48_TRAINING = False

# Enable this one switch only when the matched preflight has passed.
if RUN_B48_TRAINING:
    # Train both arms for the same fixed number of epochs; no arm is selected here.
    B48_HISTORY = train_b48_matched_pair(B48_PAIR)
    # Plot both loss trajectories together for descriptive inspection.
    plot_b48_pair_history(B48_PAIR)
    # Build the explicit paired subset comparison plus per-target top-k-change audit.
    B48_COMPARISON = evaluate_b48_matched_pair(B48_PAIR)
    # Display histories and the compact arm-level subset summary.
    show_b48_pair_results(B48_PAIR, B48_COMPARISON)
    # Build the same local-DICOM test loader used by the earlier notebook.
    TEST_LOADER = build_test_loader(TEST_PATHS, CONFIG)
    # Keep one test prediction table per matched arm rather than blending them.
    B48_TEST_PREDICTIONS = {
        experiment.arm: predict_test_set(experiment, TEST_LOADER)
        for experiment in B48_PAIR.arms()
    }
    # Review twelve unlabelled test cases from the candidate arm only for visualization.
    TEST_CASE_TABLE = show_case_examples(
        B48_PAIR.post_cross_attention_candidate,
        loader=TEST_LOADER,
        max_cases=12,
        title_prefix="Candidate test",
    )
    # Save both new models, histories, comparison JSON, scope JSON, and arm-specific predictions.
    RUN_DIRECTORY = save_b48_pair_results(
        B48_PAIR,
        B48_COMPARISON,
        test_predictions=B48_TEST_PREDICTIONS,
    )

## Memory controls and paired-run discipline

The defaults stream DICOM data, retain only four series per study, encode one
triplet at a time, and checkpoint the image encoder. The compact global
series-memory branch adds little memory compared with the 448×448 image path.
If either arm fails preflight, change only one shared `CONFIG` setting, rebuild
`B48_PAIR`, and rerun both preflights:

1. Reduce `max_series_per_study` from `4` to `3`.
2. Keep `encoder_chunk_size=1` and `gradient_checkpointing=True`.
3. Reduce `slices_per_series` from `32` to `24` only as a last resort; this
   changes the representation for both arms.
4. Keep `image_size=448` and `resize_policy="aspect_preserving_pad"` unless you
   intentionally want a different common representation.

Do not run just one arm, alter a setting between arms, inspect the first result
and then tune the second, or treat the subset comparison as the official B48
decision. The notebook records split hashes, the common initialization
fingerprint, context-gate values, and per-target top-k changes precisely to make
this sandbox comparison easy to audit without overstating it.